# 07. Software domain — final hard L5 generator

This notebook generates RU/EN Wikidata-backed multihop benchmark examples for the `software` domain.

Design goals:
- 110 examples with the target distribution `L1=10`, `L2=15`, `L3=25`, `L4=30`, `L5=30`.
- Russian and English query text for every example.
- Clean English-only `constraints` matching the query text.
- Full/strict gold collection whenever possible: examples are rejected if WDQS appears to hit the query limit or if the local gold list would be truncated.
- Schema-compatible JSONL records using the shared `BenchmarkExample` dataclass.
- Incremental generation: compatible partial output is resumed; incompatible older `software.jsonl` files are backed up before a clean target-distribution run.

Run all cells from top to bottom. The first full run may take a long time because many candidate templates are validated against Wikidata.


**v6 final L5 patch:** precise headquarters/citizenship constraints, no weak same-language/same-OS L5, near-duplicate gold overlap rejection.

In [1]:
# ============================================================
# Software domain patched generator
# ============================================================

from __future__ import annotations

import os
import re
import json
import time
import random
from pathlib import Path
from dataclasses import asdict, is_dataclass
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple, Callable

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load shared helpers

The generator uses the same `BenchmarkExample` schema and Wikidata client as the other domains. It first tries `common_helpers.py`, then falls back to executing code cells from `00_common_helpers.ipynb`.


In [2]:
from pathlib import Path as _Path

if "BenchmarkExample" not in globals():
    helper_candidates = [
        _Path("common_helpers.py"),
        _Path("../common_helpers.py"),
        _Path("00_common_helpers.ipynb"),
        _Path("../00_common_helpers.ipynb"),
    ]
    loaded = False
    for hp in helper_candidates:
        if not hp.exists():
            continue
        if hp.suffix == ".py":
            exec(hp.read_text(encoding="utf-8"), globals())
            loaded = True
            break
        if hp.suffix == ".ipynb":
            nb = json.loads(hp.read_text(encoding="utf-8"))
            for cell in nb.get("cells", []):
                if cell.get("cell_type") != "code":
                    continue
                src = "".join(cell.get("source", []))
                stripped = src.strip()
                if not stripped or stripped.startswith("!pip") or stripped.startswith("%run"):
                    continue
                exec(compile(src, str(hp), "exec"), globals())
            loaded = True
            break
    if not loaded:
        raise FileNotFoundError("Cannot find common_helpers.py or 00_common_helpers.ipynb")

print("loaded BenchmarkExample schema and Wikidata helpers")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label
loaded BenchmarkExample schema and Wikidata helpers


## Configuration and domain vocabulary

In [3]:
# =========================
# Output paths
# =========================

DOMAIN = "software"
DOMAIN_OUT_DIR = Path("out_wikidata_benchmark/domain_outputs")
DOMAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

SOFTWARE_OUT_PATH = DOMAIN_OUT_DIR / "software0.jsonl"
SOFTWARE_AUDIT_PATH = DOMAIN_OUT_DIR / "software_l5_v8_balanced_working_generation_audit.json"
SOFTWARE_CHECKPOINT_PATH = DOMAIN_OUT_DIR / "software_l5_v8_balanced_working_generation_checkpoint.json"

print("output:", SOFTWARE_OUT_PATH.resolve())
print("audit:", SOFTWARE_AUDIT_PATH.resolve())
print("checkpoint:", SOFTWARE_CHECKPOINT_PATH.resolve())

# =========================
# Generation config
# =========================

RUN_SOFTWARE_GENERATION = True

TARGET_PLAN_SOFTWARE = {
    "L1": 0,
    "L2": 0,
    "L3": 0,
    "L4": 5,
    "L5": 30,
}

SOFTWARE_SEED = 20260524
SOFTWARE_WDQS_LIMIT_DEFAULT = 1200

# Hard L5 supplement mode: generate only genuinely harder L5 into software_l5_hard.jsonl.
# Existing v0 software0.jsonl is intentionally not touched.

# Strict mode rejects broad examples where gold may be incomplete.
SOFTWARE_STRICT_FULL_GOLD = True
SOFTWARE_MIN_GOLD_BY_LEVEL = {"L1": 6, "L2": 6, "L3": 5, "L4": 4, "L5": 3}
SOFTWARE_MAX_GOLD_BY_LEVEL = {"L1": 500, "L2": 500, "L3": 500, "L4": 500, "L5": 500}
SOFTWARE_REQUESTED_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}
SOFTWARE_MAX_ATTEMPTS_BY_LEVEL = {"L1": 1800, "L2": 2200, "L3": 2600, "L4": 3400, "L5": 60000}

# Keep exact distribution by avoiding accidental appends to old software.jsonl files.
# Compatible partial runs are resumed; files with counts above the current target are backed up.
SOFTWARE_AUTOBACKUP_INCOMPATIBLE_OUTPUT = False
SOFTWARE_FORCE_FRESH_RUN = False  # resume compatible partial runs instead of wiping a good in-progress software.jsonl

# Diversity guards. They prevent one easy pattern from filling the whole file.
SOFTWARE_MAX_PER_TEMPLATE_FAMILY_TOTAL = 80
SOFTWARE_MAX_PER_TEMPLATE_FAMILY_LEVEL = 12
SOFTWARE_MAX_PER_EXACT_TEMPLATE_TOTAL = 10
SOFTWARE_MAX_PER_EXACT_TEMPLATE_LEVEL = 1

# Fail-fast / no-hang controls. These keep the notebook from spending tens of
# minutes in a low-yield phase when only a few records remain for a level.
SOFTWARE_HTTP_TIMEOUT_SECONDS = 20
SOFTWARE_WDQS_MAX_RETRIES = 2
SOFTWARE_WDQS_HARD_TIMEOUT_SECONDS = 45
SOFTWARE_NO_ACCEPT_STREAK_LIMIT_BY_LEVEL = {"L1": 350, "L2": 450, "L3": 550, "L4": 700, "L5": 15000}
SOFTWARE_PROGRESS_UPDATE_EVERY_ATTEMPTS = 10
SOFTWARE_PRUNE_EXISTING_OUTPUT = False

# Apply shorter network timeouts to the shared Wikidata client when available.
try:
    wd.timeout = SOFTWARE_HTTP_TIMEOUT_SECONDS
    wd.max_retries = SOFTWARE_WDQS_MAX_RETRIES
except Exception:
    pass

# Key QIDs. Keep constraints English, but retain QIDs internally for SPARQL only.
SOFTWARE_KINDS = {
    # Keep the broad Q7397 class only as a fallback for legacy code; the patched
    # registry never uses it directly. Broad "software" was the source of noisy
    # golds such as file formats, storage engines, duplicated Java/ZFS labels, etc.
    "software": {
        "qid": "Q7397",
        "ru_acc": "программных продуктов",
        "en_plural": "software products",
        "exclude_video_games": True,
        "broad": True,
    },
    "web_browser": {
        "qid": "Q6368",
        "ru_acc": "веб-браузеров",
        "en_plural": "web browsers",
        "exclude_video_games": False,
    },
    "operating_system": {
        "qid": "Q9135",
        "ru_acc": "операционных систем",
        "en_plural": "operating systems",
        "exclude_video_games": False,
    },
    "database_management_system": {
        "qid": "Q176165",
        "ru_acc": "систем управления базами данных",
        "en_plural": "database management systems",
        "exclude_video_games": False,
    },
    "integrated_development_environment": {
        "qid": "Q13741",
        "ru_acc": "интегрированных сред разработки",
        "en_plural": "integrated development environments",
        "exclude_video_games": False,
    },
    "software_framework": {
        "qid": "Q271680",
        "ru_acc": "программных фреймворков",
        "en_plural": "software frameworks",
        "exclude_video_games": False,
    },
    "web_server": {
        "qid": "Q11288",
        "ru_acc": "веб-серверов",
        "en_plural": "web servers",
        "exclude_video_games": False,
    },
    "text_editor": {
        "qid": "Q131212",
        "ru_acc": "текстовых редакторов",
        "en_plural": "text editors",
        "exclude_video_games": False,
    },
    "office_suite": {
        "qid": "Q207170",
        "ru_acc": "офисных пакетов",
        "en_plural": "office suites",
        "exclude_video_games": False,
    },
    "application_software": {
        "qid": "Q166142",
        "ru_acc": "прикладных программ",
        "en_plural": "application software programs",
        "exclude_video_games": True,
    },
    "programming_language": {
        "qid": "Q9143",
        "ru_acc": "языков программирования",
        "en_plural": "programming languages",
        "exclude_video_games": False,
    },
}

# Narrow answer kinds used instead of broad Q7397. Keep the list reasonably varied
# but objective; all are software subtypes with cleaner golds than generic software.
NARROW_SOFTWARE_KIND_KEYS = [
    "web_browser",
    "database_management_system",
    "integrated_development_environment",
    "software_framework",
    "web_server",
    "text_editor",
    "office_suite",
]
RUNNABLE_SOFTWARE_KIND_KEYS = [
    "web_browser",
    "integrated_development_environment",
    "text_editor",
    "office_suite",
]
DEVELOPER_FRIENDLY_KIND_KEYS = [
    "web_browser",
    "operating_system",
    "database_management_system",
    "integrated_development_environment",
    "software_framework",
    "web_server",
    "office_suite",
]

BAD_CONSTRAINT_KEYS = {
    "influenced_by_programming_language",
    "developer_also_created_programming_language_influenced_by",
}

# Broad/noisy answer kinds are kept in SOFTWARE_KINDS for backward compatibility
# and validation, but are no longer sampled for generation. They are also rejected
# by the final quality gate if a legacy template accidentally creates one.
SOFTWARE_DISABLED_KIND_KEYS = {"software", "application_software"}

# Label-based guard for software versions/releases. This catches examples like
# Firefox 3.5, MS-DOS 4.0, Microsoft Access 2007, etc. Exact version entities
# make gold lists brittle and often duplicate the same product family.
SOFTWARE_VERSION_LABEL_RE = re.compile(r"(?:\b(?:19|20)\d{2}\b|\b\d+(?:\.\d+)+\b|\b(?:version|release|edition)\b)", re.I)

# Extra hard-L5 guards. These catch release/version entities that are not caught by
# numeric labels alone (for example Android Cupcake) and prevent L5 supplements from
# duplicating L1-L4 gold lists generated in the main file.
SOFTWARE_BAD_GOLD_LABEL_SUBSTRINGS = {
    "android cupcake",
    "android donut",
    "android eclair",
    "android froyo",
    "android gingerbread",
    "android honeycomb",
    "android ice cream sandwich",
    "android jelly bean",
    "android kitkat",
    "android lollipop",
    "android marshmallow",
    "android nougat",
    "android oreo",
    "android pie",
    "firefox 3.5",
    "ms-dos 4.0",
    "microsoft access 2007",
}
SOFTWARE_BAD_GOLD_LABEL_RE = re.compile(
    r"(?:\b(?:version|release|edition|beta|alpha|rc\d*)\b|\b\d+(?:\.\d+)+\b|\bstruts\s+\d+\b)",
    re.I,
)

# Reference files are intentionally disabled for L5 supplement generation.
# We dedupe within the current output only; cross-file dedupe is done later during merge.
# In v6, reference-dedupe against older bad L5/L4 files blocked nearly all productive candidates.
SOFTWARE_REFERENCE_JSONL_PATHS = []

# Final L5 balanced-mode quality policy.
# v8 uses a two-stage generator: strong hidden license/developer bridges first,
# then a limited balanced fallback with same-language/same-OS only when combined
# with year + website/repo + direct OS/language/license constraints.
SOFTWARE_L5_FINAL_DISABLED_KINDS = set()
SOFTWARE_L5_FINAL_DISABLED_BRIDGE_KEYS = set()
SOFTWARE_L5_MAX_YEAR_SPAN = 35
SOFTWARE_L5_MIN_SUBSTANTIVE_CONSTRAINTS = 4
# Near-overlap is disabled during generation because it blocked nearly all WDQS-productive candidates.
# Exact-gold duplicate protection remains on; stronger dedupe is done during final merge.
SOFTWARE_ENABLE_NEAR_GOLD_OVERLAP_FILTER = False
SOFTWARE_MAX_GOLD_JACCARD_OVERLAP = 0.96
SOFTWARE_MAX_GOLD_CONTAINMENT_OVERLAP = 0.98
SOFTWARE_MIN_SHARED_GOLD_FOR_OVERLAP_CHECK = 4


PROGRAMMING_LANGUAGES = [
    {"en": "C", "ru": "C", "qid": "Q15777"},
    {"en": "C++", "ru": "C++", "qid": "Q2407"},
    {"en": "Python", "ru": "Python", "qid": "Q28865"},
    {"en": "Java", "ru": "Java", "qid": "Q251"},
    {"en": "JavaScript", "ru": "JavaScript", "qid": "Q2005"},
    {"en": "PHP", "ru": "PHP", "qid": "Q59"},
    {"en": "Ruby", "ru": "Ruby", "qid": "Q161053"},
    {"en": "Perl", "ru": "Perl", "qid": "Q42478"},
    {"en": "Rust", "ru": "Rust", "qid": "Q575650"},
    {"en": "Go", "ru": "Go", "qid": "Q37227"},
    {"en": "Swift", "ru": "Swift", "qid": "Q17118377"},
    {"en": "TypeScript", "ru": "TypeScript", "qid": "Q978185"},
    {"en": "Kotlin", "ru": "Kotlin", "qid": "Q3816639"},
    {"en": "Scala", "ru": "Scala", "qid": "Q460584"},
    {"en": "Haskell", "ru": "Haskell", "qid": "Q34010"},
    {"en": "R", "ru": "R", "qid": "Q206904"},
    {"en": "Lua", "ru": "Lua", "qid": "Q207316"},
    {"en": "SQL", "ru": "SQL", "qid": "Q47607"},
]

OPERATING_SYSTEMS = [
    {"en": "Microsoft Windows", "ru": "Microsoft Windows", "qid": "Q1406"},
    {"en": "Linux", "ru": "Linux", "qid": "Q388"},
    {"en": "macOS", "ru": "macOS", "qid": "Q14116"},
    {"en": "Android", "ru": "Android", "qid": "Q94"},
    {"en": "iOS", "ru": "iOS", "qid": "Q48493"},
    {"en": "Unix", "ru": "Unix", "qid": "Q11368"},
]

LICENSES = [
    {"en": "GNU General Public License", "ru": "GNU General Public License", "qid": "Q7603"},
    {"en": "MIT License", "ru": "MIT License", "qid": "Q334661"},
    {"en": "Apache License", "ru": "Apache License", "qid": "Q616526"},
    {"en": "BSD licenses", "ru": "BSD licenses", "qid": "Q191307"},
    {"en": "Mozilla Public License", "ru": "Mozilla Public License", "qid": "Q308915"},
]

COUNTRIES = [
    {"en": "United States", "ru": "США", "qid": "Q30"},
    {"en": "Germany", "ru": "Германия", "qid": "Q183"},
    {"en": "United Kingdom", "ru": "Великобритания", "qid": "Q145"},
    {"en": "France", "ru": "Франция", "qid": "Q142"},
    {"en": "Canada", "ru": "Канада", "qid": "Q16"},
    {"en": "Finland", "ru": "Финляндия", "qid": "Q33"},
    {"en": "Netherlands", "ru": "Нидерланды", "qid": "Q55"},
    {"en": "Japan", "ru": "Япония", "qid": "Q17"},
    {"en": "Brazil", "ru": "Бразилия", "qid": "Q155"},
    {"en": "Russia", "ru": "Россия", "qid": "Q159"},
    {"en": "China", "ru": "Китай", "qid": "Q148"},
]

DEVELOPERS = [
    {"en": "Microsoft", "ru": "Microsoft", "qid": "Q2283"},
    {"en": "Apple Inc.", "ru": "Apple Inc.", "qid": "Q312"},
    {"en": "Google", "ru": "Google", "qid": "Q95"},
    {"en": "IBM", "ru": "IBM", "qid": "Q37156"},
    {"en": "Oracle Corporation", "ru": "Oracle Corporation", "qid": "Q19900"},
    {"en": "Mozilla Foundation", "ru": "Mozilla Foundation", "qid": "Q55672"},
    {"en": "Apache Software Foundation", "ru": "Apache Software Foundation", "qid": "Q489709"},
    {"en": "Free Software Foundation", "ru": "Free Software Foundation", "qid": "Q48413"},
    {"en": "Adobe Inc.", "ru": "Adobe Inc.", "qid": "Q11463"},
]

# Seeds are used only to define hidden bridge constraints such as:
# "same developer as Firefox" or "same programming language as Git".
SEED_SOFTWARE = [
    {"en": "Mozilla Firefox", "ru": "Mozilla Firefox", "qid": "Q698"},
    {"en": "VLC media player", "ru": "VLC media player", "qid": "Q171477"},
    {"en": "Blender", "ru": "Blender", "qid": "Q173136"},
    {"en": "LibreOffice", "ru": "LibreOffice", "qid": "Q159557"},
    {"en": "GIMP", "ru": "GIMP", "qid": "Q8038"},
    {"en": "Chromium", "ru": "Chromium", "qid": "Q485487"},
    {"en": "Git", "ru": "Git", "qid": "Q186055"},
    {"en": "Linux kernel", "ru": "ядро Linux", "qid": "Q14579"},
    {"en": "TensorFlow", "ru": "TensorFlow", "qid": "Q21447895"},
    {"en": "PostgreSQL", "ru": "PostgreSQL", "qid": "Q192490"},
    {"en": "SQLite", "ru": "SQLite", "qid": "Q319417"},
    {"en": "WordPress", "ru": "WordPress", "qid": "Q13166"},
    {"en": "Android", "ru": "Android", "qid": "Q94"},
    {"en": "macOS", "ru": "macOS", "qid": "Q14116"},
]

SEED_PROGRAMMING_LANGUAGES = [
    {"en": "C", "ru": "C", "qid": "Q15777"},
    {"en": "C++", "ru": "C++", "qid": "Q2407"},
    {"en": "Python", "ru": "Python", "qid": "Q28865"},
    {"en": "Java", "ru": "Java", "qid": "Q251"},
    {"en": "JavaScript", "ru": "JavaScript", "qid": "Q2005"},
    {"en": "Lisp", "ru": "Lisp", "qid": "Q132874"},
    {"en": "Smalltalk", "ru": "Smalltalk", "qid": "Q828389"},
    {"en": "Perl", "ru": "Perl", "qid": "Q42478"},
    {"en": "Ruby", "ru": "Ruby", "qid": "Q161053"},
    {"en": "Haskell", "ru": "Haskell", "qid": "Q34010"},
]

YEAR_RANGES = [
    (1970, 1989), (1980, 1999), (1990, 2009), (2000, 2024),
    (1995, 2024), (2010, 2024), (2015, 2024),
]


output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/software0.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/software_l5_v8_balanced_working_generation_audit.json
checkpoint: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/software_l5_v8_balanced_working_generation_checkpoint.json


## Core helpers: SPARQL, gold collection, schema-finalization

In [4]:
_SW_QID_RE = re.compile(r"^Q\d+$")


def _sw_uri_to_qid(uri: str | None) -> Optional[str]:
    if not uri:
        return None
    try:
        return uri_to_qid(uri)
    except Exception:
        m = re.search(r"Q\d+", str(uri))
        return m.group(0) if m else None


def _sw_rows(data: dict) -> List[Dict[str, str]]:
    try:
        return rows_from_select(data)
    except Exception:
        out = []
        for b in (data or {}).get("results", {}).get("bindings", []):
            out.append({k: v.get("value") for k, v in b.items()})
        return out


def _sw_escape(s: str) -> str:
    return str(s or "").replace("\\", "\\\\").replace('"', '\\"')


def _sw_now_z() -> str:
    try:
        return utc_now_z()
    except Exception:
        import datetime as dt
        return dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")


def _sw_asdict(ex: Any) -> Dict[str, Any]:
    if is_dataclass(ex):
        return asdict(ex)
    if hasattr(ex, "__dict__"):
        return dict(ex.__dict__)
    return dict(ex)


def _sw_read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception:
                pass
    return records


def _sw_append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()


def _sw_write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()


def _sw_record_key(r: Dict[str, Any]) -> Tuple[str, str]:
    return (
        r.get("query_text_ru", ""),
        json.dumps(r.get("constraints", {}) or {}, ensure_ascii=False, sort_keys=True),
    )


def _sw_clean_constraints(c: Dict[str, Any]) -> Dict[str, Any]:
    """Keep constraints compact, English, stable and evaluator-friendly."""
    banned_exact = {
        "template_id", "template_family", "bridge_meta", "seed_qid", "seed_ru", "seed_en",
        "kind_qid", "qid", "debug", "where_lines",
    }
    out = {}
    for k, v in dict(c or {}).items():
        if k in banned_exact:
            continue
        if k.endswith("_qid") or k.endswith("_qids") or k.endswith("_ru"):
            continue
        if v is None or v is False or v == "" or v == [] or v == {}:
            continue
        if isinstance(v, dict):
            # Recursively strip internal QIDs from bridge-like metadata.
            vv = {kk: vv for kk, vv in v.items() if not kk.endswith("_qid") and not kk.endswith("_ru")}
            if vv:
                out[k] = vv
        else:
            out[k] = v
    return out


def _sw_kind_type_lines(kind: str, item_var: str = "item") -> List[str]:
    cfg = SOFTWARE_KINDS[kind]
    lines = [f"?{item_var} wdt:P31/wdt:P279* wd:{cfg['qid']} ."]
    if cfg.get("exclude_video_games"):
        # Video games are software too, but there is a separate benchmark domain for games.
        lines.append(f"FILTER NOT EXISTS {{ ?{item_var} wdt:P31/wdt:P279* wd:Q7889 . }}")
    return lines


def _sw_build_select_query(
    where_lines: List[str],
    *,
    answer_var: str = "item",
    limit: int = SOFTWARE_WDQS_LIMIT_DEFAULT,
) -> str:
    where = "\n      ".join(where_lines)
    return f"""
    SELECT DISTINCT ?{answer_var} ?{answer_var}LabelEn ?{answer_var}LabelRu WHERE {{
      {where}
      ?{answer_var} rdfs:label ?{answer_var}LabelEn FILTER(LANG(?{answer_var}LabelEn) = "en") .
      OPTIONAL {{ ?{answer_var} rdfs:label ?{answer_var}LabelRu FILTER(LANG(?{answer_var}LabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()


def _sw_build_ask_query(where_lines: List[str], *, answer_var: str = "item") -> str:
    ask_lines = [ln for ln in where_lines if "rdfs:label" not in ln]
    where = "\n      ".join(ask_lines)
    return f"""
    # WDQS-only validator. All constraints are checked directly in Wikidata.
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{answer_var})
      {where}
    }}
    """.strip()




def _sw_sparql_select_safe(query: str, use_cache: bool = True) -> dict:
    """Run WDQS with a hard wall-clock timeout, not only requests' socket timeout."""
    # requests timeout is not always a total timeout; a slow WDQS call can still
    # stall the whole generation cell. SIGALRM works in normal Jupyter/Linux main thread.
    try:
        import signal
        if not hasattr(signal, "SIGALRM"):
            return wd.sparql_select(query, use_cache=use_cache)
        old_handler = signal.getsignal(signal.SIGALRM)
        def _handler(signum, frame):
            raise TimeoutError(f"WDQS hard timeout after {SOFTWARE_WDQS_HARD_TIMEOUT_SECONDS}s")
        signal.signal(signal.SIGALRM, _handler)
        signal.setitimer(signal.ITIMER_REAL, float(SOFTWARE_WDQS_HARD_TIMEOUT_SECONDS))
        try:
            return wd.sparql_select(query, use_cache=use_cache)
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old_handler)
    except TimeoutError:
        raise

def _sw_collect_gold(
    *,
    sparql_query: str,
    answer_var: str = "item",
    wdqs_limit: int = SOFTWARE_WDQS_LIMIT_DEFAULT,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    data = _sw_sparql_select_safe(sparql_query, use_cache=True)
    rows = _sw_rows(data)

    seen = set()
    items: List[Dict[str, Any]] = []
    dropped_no_qid = 0
    dropped_no_en_label = 0
    ru_label_count = 0
    en_fallback_count = 0

    for row in rows:
        qid = _sw_uri_to_qid(row.get(answer_var))
        if not qid:
            dropped_no_qid += 1
            continue
        if qid in seen:
            continue

        label_en = (row.get(f"{answer_var}LabelEn") or "").strip()
        label_ru = (row.get(f"{answer_var}LabelRu") or "").strip()

        if not label_en or _SW_QID_RE.fullmatch(label_en):
            dropped_no_en_label += 1
            continue
        if label_ru and not _SW_QID_RE.fullmatch(label_ru):
            ru_label_count += 1
        else:
            label_ru = label_en
            en_fallback_count += 1

        seen.add(qid)
        items.append({"qid": qid, "label_ru": label_ru, "label_en": label_en})

    meta = {
        "source": "wikidata_sparql",
        "wdqs_candidate_limit": int(wdqs_limit),
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en_label,
        "label_sources": {
            "ru_label": ru_label_count,
            "en_fallback_for_ru": en_fallback_count,
        },
        "gold_may_be_incomplete_due_to_wdqs_limit": len(rows) >= int(wdqs_limit),
    }
    return items, meta


def _sw_norm_label_for_quality(label: str) -> str:
    return re.sub(r"\s+", " ", str(label or "").strip().casefold())


def _sw_duplicate_labels(labels: List[str]) -> Dict[str, int]:
    norm = [_sw_norm_label_for_quality(x) for x in labels if _sw_norm_label_for_quality(x)]
    c = Counter(norm)
    return {k: v for k, v in c.items() if v > 1}


def _sw_has_bad_constraint_key(constraints: Dict[str, Any]) -> Optional[str]:
    for k in dict(constraints or {}):
        if k in BAD_CONSTRAINT_KEYS:
            return k
    return None



def _sw_is_bad_gold_label(label: str) -> bool:
    lab = str(label or "").strip().lower()
    if not lab:
        return True
    if SOFTWARE_VERSION_LABEL_RE.search(lab):
        return True
    if globals().get("SOFTWARE_BAD_GOLD_LABEL_RE") and SOFTWARE_BAD_GOLD_LABEL_RE.search(lab):
        return True
    for bad in globals().get("SOFTWARE_BAD_GOLD_LABEL_SUBSTRINGS", set()):
        if bad in lab:
            return True
    return False


def _sw_l5_substantive_constraint_count(constraints: Dict[str, Any]) -> int:
    # Count real semantic constraints, excluding answer kind and metadata-presence
    # constraints. year_from/year_to count as one temporal constraint.
    c = dict(constraints or {})
    count = 0
    for key in [
        "programming_language_from_software",
        "operating_system_from_software",
        "license_from_software",
        "developer_from_software",
        "creator_from_programming_language",
        "developer_country",
        "creator_country",
        "programming_language",
        "operating_system",
        "license",
    ]:
        if key in c:
            count += 1
    if "year_from" in c and "year_to" in c:
        count += 1
    return count


def _sw_l5_complexity_reject_reason(record: Dict[str, Any]) -> Optional[str]:
    if record.get("complexity") != "L5":
        return None
    tid = str(record.get("template_id") or "")
    family = str(record.get("template_family") or "")
    constraints = record.get("constraints") or {}

    if tid.endswith("_base") or "_base" in tid:
        return "l5_base_template_disabled"
    if len(constraints) < 5:
        return f"l5_too_few_constraints:{len(constraints)}"

    substantive = _sw_l5_substantive_constraint_count(constraints)
    if substantive < 4:
        return f"l5_too_few_substantive_constraints:{substantive}"

    # These bridges are productive but weak. Allow them only when they are made
    # genuinely hard by a temporal constraint plus website/repository metadata.
    weak_bridge_keys = {"programming_language_from_software", "operating_system_from_software"}
    if weak_bridge_keys & set(constraints):
        has_presence = bool(constraints.get("has_official_website") or constraints.get("has_source_code_repository"))
        has_year = "year_from" in constraints and "year_to" in constraints
        if not (has_presence and has_year and substantive >= 4):
            return "l5_weak_bridge_without_year_and_presence"

    # Website/repo alone must never be the only difference from an L4-shaped task.
    if (constraints.get("has_official_website") or constraints.get("has_source_code_repository")) and substantive < 4:
        return "l5_presence_metadata_not_enough"

    return None

def _sw_gold_quality_reject_reason(gold_items: List[Dict[str, Any]], constraints: Dict[str, Any]) -> Optional[str]:
    """Hard gates for examples that are technically valid SPARQL but bad benchmark data."""
    bad_key = _sw_has_bad_constraint_key(constraints)
    if bad_key:
        return f"bad_constraint_key:{bad_key}"

    qids = [x.get("qid") for x in gold_items]
    if len(qids) != len(set(qids)):
        return "duplicate_gold_qids"

    dup_ru = _sw_duplicate_labels([x.get("label_ru", "") for x in gold_items])
    if dup_ru:
        return f"duplicate_ru_gold_labels:{sorted(dup_ru)[:5]}"

    dup_en = _sw_duplicate_labels([x.get("label_en", "") for x in gold_items])
    if dup_en:
        return f"duplicate_en_gold_labels:{sorted(dup_en)[:5]}"

    # Patched generator should not emit broad Q7397 examples anymore.
    if (constraints or {}).get("kind") == "software":
        return "broad_kind_software_is_disabled"

    kind = (constraints or {}).get("kind")
    if kind in SOFTWARE_DISABLED_KIND_KEYS:
        return f"disabled_or_broad_kind:{kind}"

    # Reject gold lists with obvious version/release entities. They are usually
    # not desired when the query asks for products/classes like browsers/OS/DBMS.
    bad_labels = []
    for x in gold_items:
        for lab in [x.get("label_en"), x.get("label_ru")]:
            if _sw_is_bad_gold_label(str(lab or "")):
                bad_labels.append(lab)
                break
    if bad_labels:
        return f"bad_or_version_like_gold_labels:{bad_labels[:5]}"

    return None


def _sw_finalize_wdqs_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    query_text_ru: str,
    query_text_en: str,
    constraints: Dict[str, Any],
    where_lines: List[str],
    requested_count: int,
    answer_var: str = "item",
    bridge_meta: Optional[Dict[str, Any]] = None,
    wdqs_limit: int = SOFTWARE_WDQS_LIMIT_DEFAULT,
) -> Optional[BenchmarkExample]:
    sparql_query = _sw_build_select_query(where_lines, answer_var=answer_var, limit=wdqs_limit)
    try:
        gold_items, collection_meta = _sw_collect_gold(
            sparql_query=sparql_query,
            answer_var=answer_var,
            wdqs_limit=wdqs_limit,
        )
    except Exception as e:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[WARN] {template_id} WDQS failed: {e}")
        return None

    if complexity == "L5":
        # L5 templates are intentionally conjunctive; allow exactly the requested
        # number of clean WDQS golds instead of requiring an extra spare answer.
        min_gold = max(int(requested_count), SOFTWARE_MIN_GOLD_BY_LEVEL.get(complexity, requested_count))
    else:
        min_gold = max(int(requested_count) + 1, SOFTWARE_MIN_GOLD_BY_LEVEL.get(complexity, requested_count))
    if len(gold_items) < min_gold:
        return None

    # Reject examples with ambiguous gold labels or bad/artificial constraints.
    quality_reason = _sw_gold_quality_reject_reason(gold_items, constraints)
    if quality_reason:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[SKIP] {template_id}: {quality_reason}")
        return None

    max_gold = SOFTWARE_MAX_GOLD_BY_LEVEL.get(complexity, 500)
    gold_truncated_by_local_limit = len(gold_items) > max_gold
    wdqs_limit_hit = bool(collection_meta.get("gold_may_be_incomplete_due_to_wdqs_limit"))

    if SOFTWARE_STRICT_FULL_GOLD and (gold_truncated_by_local_limit or wdqs_limit_hit):
        # In strict mode broad queries are rejected rather than saved with incomplete gold.
        return None

    limited_gold = gold_items[:max_gold]
    cleaned_constraints = _sw_clean_constraints(constraints)
    ask_validator = _sw_build_ask_query(where_lines, answer_var=answer_var)

    collection_meta.update({
        "constraints_are_wdqs_only": True,
        "gold_limit": max_gold,
        "gold_returned": len(limited_gold),
        "gold_total_before_limit": len(gold_items),
        "gold_truncated_by_local_limit": gold_truncated_by_local_limit,
        "template_id": template_id,
        "template_family": template_family,
    })
    if bridge_meta:
        collection_meta["bridge_meta"] = bridge_meta

    local_validator = {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "applies_after": "ask_validator_sparql",
        "filters": cleaned_constraints,
        "label_matching_used": False,
        "note": "All constraints for this software task are represented in the WDQS ASK validator; no external local validator is required.",
    }

    return BenchmarkExample(
        id=f"software_{complexity.lower()}_{idx:04d}",
        domain=DOMAIN,
        complexity=complexity,
        query_text_ru=query_text_ru,
        query_text_en=query_text_en,
        constraints=cleaned_constraints,
        requested_count=int(requested_count),
        gold_answer_qids=[x["qid"] for x in limited_gold],
        gold_answer_labels_ru=[x["label_ru"] for x in limited_gold],
        gold_answer_labels_en=[x["label_en"] for x in limited_gold],
        sparql_query=sparql_query,
        created_at=_sw_now_z(),
        is_advanced=complexity in {"L3", "L4", "L5"},
        template_id=template_id,
        template_family=template_family,
        gold_truncated=bool(gold_truncated_by_local_limit or wdqs_limit_hit),
        ask_validator_sparql=ask_validator,
        local_validator=local_validator,
        gold_collection_meta=collection_meta,
        gold_answer_imdb_ids=[],
        gold_answer_imdb_titles=[],
    )



# ============================================================
# FINAL V7 L5 QUALITY OVERRIDES
# ============================================================
# v6 was over-corrected: it used HQ-only country filters everywhere and
# reference-deduped against older supplements, so all productive templates were
# rejected before acceptance. v7 keeps the hard L5 policy, but uses unambiguous
# direct developer filters where they are more stable than country filters.

SOFTWARE_BAD_GOLD_LABEL_SUBSTRINGS = set(globals().get("SOFTWARE_BAD_GOLD_LABEL_SUBSTRINGS", set())) | {
    "android cupcake", "android donut", "android eclair", "android froyo",
    "android gingerbread", "android honeycomb", "android ice cream sandwich",
    "android jelly bean", "android kitkat", "android lollipop", "android marshmallow",
    "android nougat", "android oreo", "android pie", "android q", "android 10",
    "android 11", "android 12", "android 13", "android 14", "android 15",
    "firefox 3.5", "ms-dos 4.0", "microsoft access 2007",
}


def _sw_l5_substantive_constraint_count(constraints: Dict[str, Any]) -> int:
    """Count semantic constraints for final L5.

    `kind` and metadata-presence flags are not counted. A year range counts as one.
    Direct `developer` is counted because it is explicit and unambiguous, unlike the
    old broad `developer_country` union.
    """
    c = dict(constraints or {})
    count = 0
    for key in [
        "license_from_software",
        "developer_from_software",
        "creator_from_programming_language",
        "developer",
        "developer_headquarters_country",
        "creator_citizenship",
        "programming_language",
        "operating_system",
        "license",
    ]:
        if key in c:
            count += 1
    if "year_from" in c and "year_to" in c:
        count += 1
    return count


def _sw_l5_complexity_reject_reason(record: Dict[str, Any]) -> Optional[str]:
    if record.get("complexity") != "L5":
        return None
    tid = str(record.get("template_id") or "")
    constraints = record.get("constraints") or {}
    kind = constraints.get("kind")

    if tid.endswith("_base") or "_base" in tid:
        return "l5_base_template_disabled"
    if kind in globals().get("SOFTWARE_L5_FINAL_DISABLED_KINDS", set()):
        return f"l5_noisy_kind_disabled:{kind}"

    # Ambiguous country key from earlier notebooks is banned in final L5.
    if "developer_country" in constraints or "creator_country" in constraints:
        return "l5_ambiguous_country_constraint_disabled"

    # Weak bridges created too many L4-shaped tasks in v4/v5. v7 does not use them.
    for k in globals().get("SOFTWARE_L5_FINAL_DISABLED_BRIDGE_KEYS", set()):
        if k in constraints:
            return f"l5_weak_bridge_disabled:{k}"

    if len(constraints) < 6:
        return f"l5_too_few_constraints:{len(constraints)}"

    substantive = _sw_l5_substantive_constraint_count(constraints)
    if substantive < globals().get("SOFTWARE_L5_MIN_SUBSTANTIVE_CONSTRAINTS", 4):
        return f"l5_too_few_substantive_constraints:{substantive}"

    if "year_from" not in constraints or "year_to" not in constraints:
        return "l5_missing_temporal_constraint"
    try:
        span = int(constraints["year_to"]) - int(constraints["year_from"])
        if span < 5:
            return f"l5_year_span_too_narrow:{span}"
        if span > globals().get("SOFTWARE_L5_MAX_YEAR_SPAN", 30):
            return f"l5_year_span_too_wide:{span}"
    except Exception:
        return "l5_bad_year_range"

    # Presence flags are useful as extra filters, but never counted as substantive.
    if not (constraints.get("has_official_website") or constraints.get("has_source_code_repository")):
        return "l5_missing_website_or_repo_presence"

    return None


def _sw_gold_quality_reject_reason(gold_items: List[Dict[str, Any]], constraints: Dict[str, Any]) -> Optional[str]:
    """Hard gates for examples that are technically valid SPARQL but bad benchmark data."""
    bad_key = _sw_has_bad_constraint_key(constraints)
    if bad_key:
        return f"bad_constraint_key:{bad_key}"

    qids = [x.get("qid") for x in gold_items]
    if len(qids) != len(set(qids)):
        return "duplicate_gold_qids"

    dup_ru = _sw_duplicate_labels([x.get("label_ru", "") for x in gold_items])
    if dup_ru:
        return f"duplicate_ru_gold_labels:{sorted(dup_ru)[:5]}"

    dup_en = _sw_duplicate_labels([x.get("label_en", "") for x in gold_items])
    if dup_en:
        return f"duplicate_en_gold_labels:{sorted(dup_en)[:5]}"

    kind = (constraints or {}).get("kind")
    if kind == "software" or kind in SOFTWARE_DISABLED_KIND_KEYS:
        return f"disabled_or_broad_kind:{kind}"
    if kind in globals().get("SOFTWARE_L5_FINAL_DISABLED_KINDS", set()):
        return f"l5_noisy_kind_disabled:{kind}"

    bad_labels = []
    for x in gold_items:
        for lab in [x.get("label_en"), x.get("label_ru")]:
            if _sw_is_bad_gold_label(str(lab or "")):
                bad_labels.append(lab)
                break
    if bad_labels:
        return f"bad_or_version_like_gold_labels:{bad_labels[:5]}"

    return None



## SPARQL condition builders and query text helpers

In [5]:
def _sw_year_lines(year_from: int, year_to: int, item_var: str = "item", date_var: str = "date") -> List[str]:
    return [
        f"?{item_var} (wdt:P571|wdt:P577) ?{date_var} .",
        f"FILTER(YEAR(?{date_var}) >= {int(year_from)} && YEAR(?{date_var}) <= {int(year_to)}) .",
    ]


def _sw_inception_year_lines(year_from: int, year_to: int, item_var: str = "item", date_var: str = "inceptionDate") -> List[str]:
    return [
        f"?{item_var} wdt:P571 ?{date_var} .",
        f"FILTER(YEAR(?{date_var}) >= {int(year_from)} && YEAR(?{date_var}) <= {int(year_to)}) .",
    ]


def _sw_programming_language_line(lang: Dict[str, str], item_var: str = "item") -> str:
    return f"?{item_var} wdt:P277 wd:{lang['qid']} ."


def _sw_operating_system_line(os_item: Dict[str, str], item_var: str = "item") -> str:
    return f"?{item_var} wdt:P306 wd:{os_item['qid']} ."


def _sw_license_line(license_item: Dict[str, str], item_var: str = "item") -> str:
    return f"?{item_var} wdt:P275 wd:{license_item['qid']} ."


def _sw_developer_line(dev: Dict[str, str], item_var: str = "item") -> str:
    return f"?{item_var} wdt:P178 wd:{dev['qid']} ."


def _sw_has_official_website_line(item_var: str = "item") -> str:
    return f"?{item_var} wdt:P856 ?officialWebsite ."


def _sw_has_source_repo_line(item_var: str = "item") -> str:
    return f"?{item_var} wdt:P1324 ?sourceCodeRepository ."


def _sw_developer_country_lines(country: Dict[str, str], item_var: str = "item", developer_var: str = "developer") -> List[str]:
    cq = country["qid"]
    return [
        f"?{item_var} wdt:P178 ?{developer_var} .",
        "{ "
        f"?{developer_var} wdt:P17 wd:{cq} . "
        "} UNION { "
        f"?{developer_var} wdt:P27 wd:{cq} . "
        "} UNION { "
        f"?{developer_var} wdt:P495 wd:{cq} . "
        "} UNION { "
        f"?{developer_var} wdt:P159/wdt:P17 wd:{cq} . "
        "}",
    ]


def _sw_creator_country_lines(country: Dict[str, str], item_var: str = "item", creator_var: str = "creator") -> List[str]:
    cq = country["qid"]
    return [
        f"?{item_var} (wdt:P178|wdt:P287|wdt:P170) ?{creator_var} .",
        "{ "
        f"?{creator_var} wdt:P17 wd:{cq} . "
        "} UNION { "
        f"?{creator_var} wdt:P27 wd:{cq} . "
        "} UNION { "
        f"?{creator_var} wdt:P495 wd:{cq} . "
        "} UNION { "
        f"?{creator_var} wdt:P159/wdt:P17 wd:{cq} . "
        "}",
    ]


def _sw_same_developer_as_seed_lines(seed: Dict[str, str], item_var: str = "item") -> List[str]:
    return [
        f"BIND(wd:{seed['qid']} AS ?seedSoftware) .",
        "?seedSoftware wdt:P178 ?bridgeDeveloper .",
        f"?{item_var} wdt:P178 ?bridgeDeveloper .",
        f"FILTER(?{item_var} != ?seedSoftware) .",
    ]


def _sw_same_programming_language_as_seed_lines(seed: Dict[str, str], item_var: str = "item") -> List[str]:
    return [
        f"BIND(wd:{seed['qid']} AS ?seedSoftware) .",
        "?seedSoftware wdt:P277 ?bridgeProgrammingLanguage .",
        f"?{item_var} wdt:P277 ?bridgeProgrammingLanguage .",
        f"FILTER(?{item_var} != ?seedSoftware) .",
    ]


def _sw_same_license_as_seed_lines(seed: Dict[str, str], item_var: str = "item") -> List[str]:
    return [
        f"BIND(wd:{seed['qid']} AS ?seedSoftware) .",
        "?seedSoftware wdt:P275 ?bridgeLicense .",
        f"?{item_var} wdt:P275 ?bridgeLicense .",
        f"FILTER(?{item_var} != ?seedSoftware) .",
    ]


def _sw_same_operating_system_as_seed_lines(seed: Dict[str, str], item_var: str = "item") -> List[str]:
    return [
        f"BIND(wd:{seed['qid']} AS ?seedSoftware) .",
        "?seedSoftware wdt:P306 ?bridgeOperatingSystem .",
        f"?{item_var} wdt:P306 ?bridgeOperatingSystem .",
        f"FILTER(?{item_var} != ?seedSoftware) .",
    ]


def _sw_programming_language_influenced_by_lines(seed_lang: Dict[str, str], item_var: str = "item") -> List[str]:
    return [f"?{item_var} wdt:P737 wd:{seed_lang['qid']} ."]


def _sw_same_creator_as_language_lines(seed_lang: Dict[str, str], item_var: str = "item") -> List[str]:
    return [
        f"BIND(wd:{seed_lang['qid']} AS ?seedLanguage) .",
        "?seedLanguage (wdt:P178|wdt:P287|wdt:P170) ?bridgeCreator .",
        f"?{item_var} (wdt:P178|wdt:P287|wdt:P170) ?bridgeCreator .",
        f"FILTER(?{item_var} != ?seedLanguage) .",
    ]


def _sw_developer_also_created_language_lines(seed_lang: Dict[str, str], item_var: str = "item") -> List[str]:
    return [
        f"?{item_var} wdt:P178 ?bridgeDeveloper .",
        "?bridgeLanguage wdt:P31/wdt:P279* wd:Q9143 .",
        "?bridgeLanguage (wdt:P178|wdt:P287|wdt:P170) ?bridgeDeveloper .",
        f"?bridgeLanguage wdt:P737 wd:{seed_lang['qid']} .",
    ]


def _sw_requested_count(complexity: str) -> int:
    return SOFTWARE_REQUESTED_BY_LEVEL.get(complexity, 3)




SOFTWARE_RU_COUNT_FORMS_2_4 = {
    "web_browser": "веб-браузера",
    "operating_system": "операционные системы",
    "database_management_system": "системы управления базами данных",
    "integrated_development_environment": "интегрированные среды разработки",
    "software_framework": "программных фреймворка",
    "web_server": "веб-сервера",
    "text_editor": "текстовых редактора",
    "office_suite": "офисных пакета",
    "application_software": "прикладные программы",
    "programming_language": "языка программирования",
}
SOFTWARE_RU_FEM_PLURAL_KINDS = {"operating_system", "integrated_development_environment", "application_software"}


def _sw_ru_kind_phrase(kind: str, requested_count: int) -> str:
    if requested_count % 100 not in {12, 13, 14} and requested_count % 10 in {2, 3, 4}:
        return SOFTWARE_RU_COUNT_FORMS_2_4.get(kind, SOFTWARE_KINDS[kind]["ru_acc"])
    return SOFTWARE_KINDS[kind]["ru_acc"]


def _sw_adjust_ru_parts_for_count(kind: str, requested_count: int, ru_parts: List[str]) -> List[str]:
    # For feminine plural heads like "4 операционные системы" / "4 интегрированные среды",
    # convert the most common participles from genitive plural to nominative/accusative plural.
    if not (requested_count % 100 not in {12, 13, 14} and requested_count % 10 in {2, 3, 4} and kind in SOFTWARE_RU_FEM_PLURAL_KINDS):
        return list(ru_parts)
    repl = {
        "написанных": "написанные",
        "работающих": "работающие",
        "распространяемых": "распространяемые",
        "созданных": "созданные",
        "имеющих": "имеющие",
        "у которых": "у которых",
    }
    out = []
    for p in ru_parts:
        pp = str(p)
        for a, b in repl.items():
            if pp.startswith(a):
                pp = b + pp[len(a):]
                break
        out.append(pp)
    return out


def _sw_head(kind: str, requested_count: int) -> Tuple[str, str]:
    cfg = SOFTWARE_KINDS[kind]
    return (
        f"Назови {requested_count} {_sw_ru_kind_phrase(kind, requested_count)}",
        f"Name {requested_count} {cfg['en_plural']}",
    )


def _sw_finish_text(kind: str, requested_count: int, ru_parts: List[str], en_parts: List[str]) -> Tuple[str, str]:
    ru_head, en_head = _sw_head(kind, requested_count)
    ru_parts = _sw_adjust_ru_parts_for_count(kind, requested_count, ru_parts)
    ru = ru_head
    en = en_head
    if ru_parts:
        ru += ", " + ", ".join(ru_parts)
    if en_parts:
        en += " that " + ", and ".join(en_parts)
    return ru.rstrip(" .") + ".", en.rstrip(" .") + "."


def _sw_year_phrase(year_from: int, year_to: int) -> Tuple[str, str]:
    return f"созданных или выпущенных в {year_from}–{year_to} годах", f"were created or released between {year_from} and {year_to}"


def _sw_inception_year_phrase(year_from: int, year_to: int) -> Tuple[str, str]:
    return f"созданных в {year_from}–{year_to} годах", f"were created between {year_from} and {year_to}"


def _sw_bridge_meta(
    *,
    bridge: str,
    constraint_key: str,
    property_id: Optional[str] = None,
    property_chain: Optional[List[str]] = None,
    seed: Optional[Dict[str, Any]] = None,
    semantics: str = "shared_or_related_value",
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    out = {
        "bridge": bridge,
        "constraint_key": constraint_key,
        "semantics": semantics,
        "intermediate_value_hidden_in_query": True,
    }
    if property_id:
        out["wikidata_property"] = property_id
    if property_chain:
        out["wikidata_property_chain"] = property_chain
    if seed:
        out.update({
            "seed_label_en": seed.get("en"),
            "seed_label_ru": seed.get("ru"),
        })
    if extra:
        out.update(extra)
    return out


def _sw_pick(rng: random.Random, items: List[Dict[str, str]]) -> Dict[str, str]:
    return rng.choice(items)



def _sw_pick_kind_key(rng: random.Random, keys: List[str]) -> str:
    return rng.choice(list(keys))


def _sw_pick_narrow_kind(rng: random.Random) -> str:
    return _sw_pick_kind_key(rng, NARROW_SOFTWARE_KIND_KEYS)


def _sw_pick_runnable_kind(rng: random.Random) -> str:
    return _sw_pick_kind_key(rng, RUNNABLE_SOFTWARE_KIND_KEYS)


def _sw_pick_developer_friendly_kind(rng: random.Random) -> str:
    return _sw_pick_kind_key(rng, DEVELOPER_FRIENDLY_KIND_KEYS)


def _sw_license_phrase(lic: Dict[str, str]) -> Tuple[str, str]:
    return f"распространяемых под лицензией {lic['ru']}", f"are released under the {lic['en']}"


def _sw_developer_country_phrase(country: Dict[str, str]) -> Tuple[str, str]:
    return f"разработчик которых связан с {country['ru']}", f"have a developer associated with {country['en']}"


## L3-L5 template builders

In [6]:


# ============================================================
# L1-L2 template builders
# ============================================================
# Patched: no broad kind="software" and no subjective Wikidata P737 "influenced by".
# L1/L2 still remain simple, but answer kinds are narrow and constraints are objective.


def _sw_tpl_l1_narrow_by_developer(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    kind = _sw_pick_developer_friendly_kind(rng)
    dev = _sw_pick(rng, DEVELOPERS)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_developer_line(dev))
    constraints = {"kind": kind, "developer": dev["en"]}
    ru_parts = [f"разработанных {dev['ru']}"]
    en_parts = [f"were developed by {dev['en']}"]
    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l1_narrow_developer", template_family="simple_narrow_developer",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l1_narrow_by_language(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    kind = _sw_pick_narrow_kind(rng)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_programming_language_line(lang))
    constraints = {"kind": kind, "programming_language": lang["en"]}
    ru_parts = [f"написанных на {lang['ru']}"]
    en_parts = [f"are written in {lang['en']}"]
    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l1_narrow_language", template_family="simple_narrow_language",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l1_narrow_by_license(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    kind = _sw_pick_narrow_kind(rng)
    lic = _sw_pick(rng, LICENSES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_license_line(lic))
    constraints = {"kind": kind, "license": lic["en"]}
    ru_l, en_l = _sw_license_phrase(lic)
    q_ru, q_en = _sw_finish_text(kind, requested, [ru_l], [en_l])
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l1_narrow_license", template_family="simple_narrow_license",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l1_web_browser_by_os(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    kind = "web_browser"
    os_item = _sw_pick(rng, OPERATING_SYSTEMS)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_operating_system_line(os_item))
    constraints = {"kind": kind, "operating_system": os_item["en"]}
    ru_parts = [f"работающих на {os_item['ru']}"]
    en_parts = [f"run on {os_item['en']}"]
    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l1_browser_os", template_family="simple_browser_os",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l1_operating_system_by_language(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L1":
        return None
    kind = "operating_system"
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_programming_language_line(lang))
    constraints = {"kind": kind, "programming_language": lang["en"]}
    q_ru, q_en = _sw_finish_text(kind, requested, [f"написанных на {lang['ru']}"] , [f"are written in {lang['en']}"])
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l1_os_language", template_family="simple_os_language",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l2_narrow_language_license(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    kind = _sw_pick_narrow_kind(rng)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    lic = _sw_pick(rng, LICENSES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_programming_language_line(lang))
    where.append(_sw_license_line(lic))
    constraints = {"kind": kind, "programming_language": lang["en"], "license": lic["en"]}
    ru_l, en_l = _sw_license_phrase(lic)
    ru_parts = [f"написанных на {lang['ru']}", ru_l]
    en_parts = [f"are written in {lang['en']}", en_l]
    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l2_narrow_language_license", template_family="narrow_language_license",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l2_narrow_language_year(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    kind = _sw_pick_narrow_kind(rng)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    year_from, year_to = rng.choice(YEAR_RANGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_programming_language_line(lang))
    where += _sw_year_lines(year_from, year_to)
    constraints = {"kind": kind, "programming_language": lang["en"], "year_from": year_from, "year_to": year_to}
    ru_y, en_y = _sw_year_phrase(year_from, year_to)
    ru_parts = [f"написанных на {lang['ru']}", ru_y]
    en_parts = [f"are written in {lang['en']}", en_y]
    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l2_narrow_language_year", template_family="narrow_language_year",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l2_narrow_devcountry_license(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    kind = _sw_pick_developer_friendly_kind(rng)
    country = _sw_pick(rng, COUNTRIES)
    lic = _sw_pick(rng, LICENSES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_developer_country_lines(country)
    where.append(_sw_license_line(lic))
    ru_c, en_c = _sw_developer_country_phrase(country)
    ru_l, en_l = _sw_license_phrase(lic)
    constraints = {"kind": kind, "developer_country": country["en"], "license": lic["en"]}
    q_ru, q_en = _sw_finish_text(kind, requested, [ru_c, ru_l], [en_c, en_l])
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l2_narrow_devcountry_license", template_family="narrow_developer_country_license",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="developer_country", constraint_key="developer_country", property_chain=["P178", "P17/P27/P495/P159/P17"]),
    )


def _sw_tpl_l2_runnable_os_license(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    kind = _sw_pick_runnable_kind(rng)
    os_item = _sw_pick(rng, OPERATING_SYSTEMS)
    lic = _sw_pick(rng, LICENSES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_operating_system_line(os_item))
    where.append(_sw_license_line(lic))
    ru_l, en_l = _sw_license_phrase(lic)
    constraints = {"kind": kind, "operating_system": os_item["en"], "license": lic["en"]}
    q_ru, q_en = _sw_finish_text(kind, requested, [f"работающих на {os_item['ru']}", ru_l], [f"run on {os_item['en']}", en_l])
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l2_runnable_os_license", template_family="runnable_os_license",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l2_web_browser_os_language(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    kind = "web_browser"
    os_item = _sw_pick(rng, OPERATING_SYSTEMS)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_operating_system_line(os_item))
    where.append(_sw_programming_language_line(lang))
    constraints = {"kind": kind, "operating_system": os_item["en"], "programming_language": lang["en"]}
    q_ru, q_en = _sw_finish_text(kind, requested, [f"работающих на {os_item['ru']}", f"написанных на {lang['ru']}"], [f"run on {os_item['en']}", f"are written in {lang['en']}"])
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l2_browser_os_language", template_family="browser_os_language",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_l2_os_language_year(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    kind = "operating_system"
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    year_from, year_to = rng.choice(YEAR_RANGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_programming_language_line(lang))
    where += _sw_year_lines(year_from, year_to)
    constraints = {"kind": kind, "programming_language": lang["en"], "year_from": year_from, "year_to": year_to}
    ru_y, en_y = _sw_year_phrase(year_from, year_to)
    q_ru, q_en = _sw_finish_text(kind, requested, [f"написанных на {lang['ru']}", ru_y], [f"are written in {lang['en']}", en_y])
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="software_l2_os_language_year", template_family="os_language_year",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_same_developer_language(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = _sw_pick_developer_friendly_kind(rng)
    seed = _sw_pick(rng, SEED_SOFTWARE)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_same_developer_as_seed_lines(seed)
    where.append(_sw_programming_language_line(lang))

    constraints = {
        "kind": kind,
        "developer_from_software": seed["en"],
        "programming_language": lang["en"],
    }

    ru_parts = [f"разработанных тем же разработчиком, что и «{seed['ru']}»", f"написанных на {lang['ru']}"]
    en_parts = [f"were developed by the same developer as \"{seed['en']}\"", f"are written in {lang['en']}"]

    template_id = "software_same_developer_language"
    if complexity in {"L4", "L5"}:
        if rng.random() < 0.55:
            year_from, year_to = rng.choice(YEAR_RANGES)
            where += _sw_year_lines(year_from, year_to)
            constraints.update({"year_from": year_from, "year_to": year_to})
            ru_y, en_y = _sw_year_phrase(year_from, year_to)
            ru_parts.append(ru_y); en_parts.append(en_y)
            template_id += "_year"
        else:
            lic = _sw_pick(rng, LICENSES)
            where.append(_sw_license_line(lic))
            constraints["license"] = lic["en"]
            ru_parts.append(f"распространяемых под лицензией {lic['ru']}")
            en_parts.append(f"are released under the {lic['en']}")
            template_id += "_license"

    if complexity == "L5":
        if rng.random() < 0.5:
            where.append(_sw_has_source_repo_line())
            constraints["has_source_code_repository"] = True
            ru_parts.append("у которых указан репозиторий исходного кода")
            en_parts.append("have a source code repository recorded")
            template_id += "_repo"
        else:
            where.append(_sw_has_official_website_line())
            constraints["has_official_website"] = True
            ru_parts.append("у которых указан официальный сайт")
            en_parts.append("have an official website recorded")
            template_id += "_website"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="same_developer",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where,
        requested_count=requested, bridge_meta=_sw_bridge_meta(bridge="developer", constraint_key="developer_from_software", property_id="P178", seed=seed),
    )


def _sw_tpl_same_language_os_license(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = _sw_pick_runnable_kind(rng)
    seed = _sw_pick(rng, SEED_SOFTWARE)
    os_item = _sw_pick(rng, OPERATING_SYSTEMS)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_same_programming_language_as_seed_lines(seed)
    where.append(_sw_operating_system_line(os_item))

    constraints = {
        "kind": kind,
        "programming_language_from_software": seed["en"],
        "operating_system": os_item["en"],
    }
    ru_parts = [f"написанных хотя бы на одном из тех же языков программирования, что и «{seed['ru']}»", f"работающих на {os_item['ru']}"]
    en_parts = [f"are written in at least one of the same programming languages as \"{seed['en']}\"", f"run on {os_item['en']}"]

    template_id = "software_same_language_os"
    if complexity in {"L4", "L5"}:
        lic = _sw_pick(rng, LICENSES)
        where.append(_sw_license_line(lic))
        constraints["license"] = lic["en"]
        ru_parts.append(f"распространяемых под лицензией {lic['ru']}")
        en_parts.append(f"are released under the {lic['en']}")
        template_id += "_license"

    if complexity == "L5":
        year_from, year_to = rng.choice(YEAR_RANGES)
        where += _sw_year_lines(year_from, year_to)
        constraints.update({"year_from": year_from, "year_to": year_to})
        ru_y, en_y = _sw_year_phrase(year_from, year_to)
        ru_parts.append(ru_y); en_parts.append(en_y)
        if rng.random() < 0.45:
            where.append(_sw_has_source_repo_line())
            constraints["has_source_code_repository"] = True
            ru_parts.append("у которых указан репозиторий исходного кода")
            en_parts.append("have a source code repository recorded")
        template_id += "_year"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="same_programming_language",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where,
        requested_count=requested, bridge_meta=_sw_bridge_meta(bridge="programming_language", constraint_key="programming_language_from_software", property_id="P277", seed=seed),
    )


def _sw_tpl_developer_country_language_year(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = _sw_pick_developer_friendly_kind(rng)
    country = _sw_pick(rng, COUNTRIES)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    year_from, year_to = rng.choice(YEAR_RANGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_developer_country_lines(country)
    where.append(_sw_programming_language_line(lang))
    where += _sw_year_lines(year_from, year_to)

    constraints = {
        "kind": kind,
        "developer_country": country["en"],
        "programming_language": lang["en"],
        "year_from": year_from,
        "year_to": year_to,
    }
    ru_y, en_y = _sw_year_phrase(year_from, year_to)
    ru_parts = [f"у разработчика, связанного со страной: {country['ru']}", f"написанных на {lang['ru']}", ru_y]
    en_parts = [f"have a developer associated with {country['en']}", f"are written in {lang['en']}", en_y]
    template_id = "software_developer_country_language_year"

    if complexity in {"L4", "L5"}:
        if rng.random() < 0.5:
            os_item = _sw_pick(rng, OPERATING_SYSTEMS)
            where.append(_sw_operating_system_line(os_item))
            constraints["operating_system"] = os_item["en"]
            ru_parts.append(f"работающих на {os_item['ru']}")
            en_parts.append(f"run on {os_item['en']}")
            template_id += "_os"
        else:
            lic = _sw_pick(rng, LICENSES)
            where.append(_sw_license_line(lic))
            constraints["license"] = lic["en"]
            ru_parts.append(f"распространяемых под лицензией {lic['ru']}")
            en_parts.append(f"are released under the {lic['en']}")
            template_id += "_license"

    if complexity == "L5":
        where.append(_sw_has_official_website_line())
        constraints["has_official_website"] = True
        ru_parts.append("у которых указан официальный сайт")
        en_parts.append("have an official website recorded")
        template_id += "_website"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="developer_country_language",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="developer_country", constraint_key="developer_country", property_chain=["P178", "P17/P27/P495/P159/P17"]),
    )


def _sw_tpl_license_language_os_devcountry(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = _sw_pick_runnable_kind(rng)
    lic = _sw_pick(rng, LICENSES)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    os_item = _sw_pick(rng, OPERATING_SYSTEMS)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where.append(_sw_license_line(lic))
    where.append(_sw_programming_language_line(lang))
    where.append(_sw_operating_system_line(os_item))

    constraints = {
        "kind": kind,
        "license": lic["en"],
        "programming_language": lang["en"],
        "operating_system": os_item["en"],
    }
    ru_parts = [f"распространяемых под лицензией {lic['ru']}", f"написанных на {lang['ru']}", f"работающих на {os_item['ru']}"]
    en_parts = [f"are released under the {lic['en']}", f"are written in {lang['en']}", f"run on {os_item['en']}"]
    template_id = "software_license_language_os"

    if complexity in {"L4", "L5"}:
        country = _sw_pick(rng, COUNTRIES)
        where += _sw_developer_country_lines(country, developer_var="developer2")
        constraints["developer_country"] = country["en"]
        ru_parts.append(f"у разработчика, связанного со страной: {country['ru']}")
        en_parts.append(f"have a developer associated with {country['en']}")
        template_id += "_devcountry"

    if complexity == "L5":
        year_from, year_to = rng.choice(YEAR_RANGES)
        where += _sw_year_lines(year_from, year_to)
        constraints.update({"year_from": year_from, "year_to": year_to})
        ru_y, en_y = _sw_year_phrase(year_from, year_to)
        ru_parts.append(ru_y); en_parts.append(en_y)
        if rng.random() < 0.5:
            where.append(_sw_has_source_repo_line())
            constraints["has_source_code_repository"] = True
            ru_parts.append("у которых указан репозиторий исходного кода")
            en_parts.append("have a source code repository recorded")
        template_id += "_year"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="license_language_os",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
    )


def _sw_tpl_operating_system_devcountry_language(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = "operating_system"
    country = _sw_pick(rng, COUNTRIES)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_developer_country_lines(country)
    where.append(_sw_programming_language_line(lang))

    constraints = {
        "kind": kind,
        "developer_country": country["en"],
        "programming_language": lang["en"],
    }
    ru_parts = [f"у разработчика, связанного со страной: {country['ru']}", f"написанных на {lang['ru']}"]
    en_parts = [f"have a developer associated with {country['en']}", f"are written in {lang['en']}"]
    template_id = "software_os_devcountry_language"

    if complexity in {"L4", "L5"}:
        if rng.random() < 0.55:
            lic = _sw_pick(rng, LICENSES)
            where.append(_sw_license_line(lic))
            constraints["license"] = lic["en"]
            ru_parts.append(f"распространяемых под лицензией {lic['ru']}")
            en_parts.append(f"are released under the {lic['en']}")
            template_id += "_license"
        else:
            year_from, year_to = rng.choice(YEAR_RANGES)
            where += _sw_year_lines(year_from, year_to)
            constraints.update({"year_from": year_from, "year_to": year_to})
            ru_y, en_y = _sw_year_phrase(year_from, year_to)
            ru_parts.append(ru_y); en_parts.append(en_y)
            template_id += "_year"

    if complexity == "L5":
        where.append(_sw_has_official_website_line())
        constraints["has_official_website"] = True
        ru_parts.append("у которых указан официальный сайт")
        en_parts.append("have an official website recorded")
        template_id += "_website"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="operating_system",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="developer_country", constraint_key="developer_country", property_chain=["P178", "P17/P27/P495/P159/P17"]),
    )


def _sw_tpl_operating_system_same_developer_as_seed(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    kind = "operating_system"
    seed = _sw_pick(rng, SEED_SOFTWARE)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_same_developer_as_seed_lines(seed)
    where.append(_sw_programming_language_line(lang))

    constraints = {
        "kind": kind,
        "developer_from_software": seed["en"],
        "programming_language": lang["en"],
    }
    ru_parts = [f"разработанных тем же разработчиком, что и «{seed['ru']}»", f"написанных на {lang['ru']}"]
    en_parts = [f"were developed by the same developer as \"{seed['en']}\"", f"are written in {lang['en']}"]
    template_id = "software_os_same_developer_language"

    if complexity == "L5":
        lic = _sw_pick(rng, LICENSES)
        where.append(_sw_license_line(lic))
        constraints["license"] = lic["en"]
        ru_parts.append(f"распространяемых под лицензией {lic['ru']}")
        en_parts.append(f"are released under the {lic['en']}")
        template_id += "_license"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="operating_system_same_developer",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="developer", constraint_key="developer_from_software", property_id="P178", seed=seed),
    )


def _sw_tpl_web_browser_devcountry_language_license(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = "web_browser"
    country = _sw_pick(rng, COUNTRIES)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_developer_country_lines(country)
    where.append(_sw_programming_language_line(lang))

    constraints = {
        "kind": kind,
        "developer_country": country["en"],
        "programming_language": lang["en"],
    }
    ru_parts = [f"у разработчика, связанного со страной: {country['ru']}", f"написанных на {lang['ru']}"]
    en_parts = [f"have a developer associated with {country['en']}", f"are written in {lang['en']}"]
    template_id = "software_browser_devcountry_language"

    if complexity in {"L4", "L5"}:
        lic = _sw_pick(rng, LICENSES)
        where.append(_sw_license_line(lic))
        constraints["license"] = lic["en"]
        ru_parts.append(f"распространяемых под лицензией {lic['ru']}")
        en_parts.append(f"are released under the {lic['en']}")
        template_id += "_license"

    if complexity == "L5":
        os_item = _sw_pick(rng, OPERATING_SYSTEMS)
        where.append(_sw_operating_system_line(os_item))
        constraints["operating_system"] = os_item["en"]
        ru_parts.append(f"работающих на {os_item['ru']}")
        en_parts.append(f"run on {os_item['en']}")
        template_id += "_os"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="web_browser",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="developer_country", constraint_key="developer_country", property_chain=["P178", "P17/P27/P495/P159/P17"]),
    )


def _sw_tpl_programming_language_same_creator_as_seed(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    kind = "programming_language"
    seed_lang = _sw_pick(rng, SEED_PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_same_creator_as_language_lines(seed_lang)

    constraints = {
        "kind": kind,
        "creator_from_programming_language": seed_lang["en"],
    }
    ru_parts = [f"созданных тем же создателем или разработчиком, что и {seed_lang['ru']}"]
    en_parts = [f"were created or developed by the same creator/developer as {seed_lang['en']}"]
    template_id = "software_language_same_creator"

    if complexity in {"L4", "L5"}:
        country = _sw_pick(rng, COUNTRIES)
        where += _sw_creator_country_lines(country, creator_var="creator_lang")
        constraints["creator_country"] = country["en"]
        ru_parts.append(f"у создателя или разработчика, связанного со страной: {country['ru']}")
        en_parts.append(f"have a creator or developer associated with {country['en']}")
        template_id += "_creatorcountry"

    if complexity == "L5":
        year_from, year_to = rng.choice(YEAR_RANGES)
        where += _sw_inception_year_lines(year_from, year_to)
        constraints.update({"year_from": year_from, "year_to": year_to})
        ru_y, en_y = _sw_inception_year_phrase(year_from, year_to)
        ru_parts.append(ru_y); en_parts.append(en_y)
        template_id += "_year"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="programming_language_same_creator",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="creator", constraint_key="creator_from_programming_language", property_chain=["P178", "P287", "P170"], seed=seed_lang),
    )


def _sw_tpl_same_license_as_seed_language_country(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = _sw_pick_developer_friendly_kind(rng)
    seed = _sw_pick(rng, SEED_SOFTWARE)
    lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_same_license_as_seed_lines(seed)
    where.append(_sw_programming_language_line(lang))

    constraints = {
        "kind": kind,
        "license_from_software": seed["en"],
        "programming_language": lang["en"],
    }
    ru_parts = [f"имеющих хотя бы одну из тех же лицензий, что и «{seed['ru']}»", f"написанных на {lang['ru']}"]
    en_parts = [f"share at least one license with \"{seed['en']}\"", f"are written in {lang['en']}"]
    template_id = "software_same_license_language"

    if complexity in {"L4", "L5"}:
        country = _sw_pick(rng, COUNTRIES)
        where += _sw_developer_country_lines(country, developer_var="developer3")
        constraints["developer_country"] = country["en"]
        ru_parts.append(f"у разработчика, связанного со страной: {country['ru']}")
        en_parts.append(f"have a developer associated with {country['en']}")
        template_id += "_devcountry"

    if complexity == "L5":
        where.append(_sw_has_source_repo_line())
        constraints["has_source_code_repository"] = True
        ru_parts.append("у которых указан репозиторий исходного кода")
        en_parts.append("have a source code repository recorded")
        template_id += "_repo"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="same_license",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="license", constraint_key="license_from_software", property_id="P275", seed=seed),
    )


def _sw_tpl_same_os_as_seed_license_language(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    kind = _sw_pick_runnable_kind(rng)
    seed = _sw_pick(rng, SEED_SOFTWARE)
    lic = _sw_pick(rng, LICENSES)
    requested = _sw_requested_count(complexity)

    where = _sw_kind_type_lines(kind)
    where += _sw_same_operating_system_as_seed_lines(seed)
    where.append(_sw_license_line(lic))

    constraints = {
        "kind": kind,
        "operating_system_from_software": seed["en"],
        "license": lic["en"],
    }
    ru_parts = [f"работающих хотя бы на одной из тех же операционных систем, что и «{seed['ru']}»", f"распространяемых под лицензией {lic['ru']}"]
    en_parts = [f"run on at least one of the same operating systems as \"{seed['en']}\"", f"are released under the {lic['en']}"]
    template_id = "software_same_os_license"

    if complexity in {"L4", "L5"}:
        lang = _sw_pick(rng, PROGRAMMING_LANGUAGES)
        where.append(_sw_programming_language_line(lang))
        constraints["programming_language"] = lang["en"]
        ru_parts.append(f"написанных на {lang['ru']}")
        en_parts.append(f"are written in {lang['en']}")
        template_id += "_language"

    if complexity == "L5":
        year_from, year_to = rng.choice(YEAR_RANGES)
        where += _sw_year_lines(year_from, year_to)
        constraints.update({"year_from": year_from, "year_to": year_to})
        ru_y, en_y = _sw_year_phrase(year_from, year_to)
        ru_parts.append(ru_y); en_parts.append(en_y)
        template_id += "_year"

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    return _sw_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id=template_id, template_family="same_operating_system",
        query_text_ru=q_ru, query_text_en=q_en, constraints=constraints, where_lines=where, requested_count=requested,
        bridge_meta=_sw_bridge_meta(bridge="operating_system", constraint_key="operating_system_from_software", property_id="P306", seed=seed),
    )


## Template registry and generation loop

In [7]:

# ============================================================
# Final hard/balanced L5 template builder — V8 working version
# ============================================================
# Why v7 undergenerated:
# - it removed weak bridges completely and had too few WDQS-productive strong combos;
# - near-overlap filtering rejected the remaining productive variants;
# - random retries kept hitting dead candidates.
#
# v8 fixes that by:
# - trying each concrete L5 candidate at most once unless it is accepted/rejected externally;
# - using a larger pool of strong candidates first;
# - adding a limited balanced fallback for same-language/same-OS patterns only when they
#   also have direct OS/language/license + year + website/repo filters;
# - keeping exact-gold duplicate protection, but leaving near-deduplication to merge time.


def _sw_by_en(pool: List[Dict[str, str]], en: str) -> Dict[str, str]:
    for x in pool:
        if x.get("en") == en:
            return x
    raise KeyError(en)


def _sw_developer_hq_country_lines(country: Dict[str, str], item_var: str = "item", developer_var: str = "developer") -> List[str]:
    cq = country["qid"]
    return [
        f"?{item_var} wdt:P178 ?{developer_var} .",
        f"?{developer_var} wdt:P159/wdt:P17 wd:{cq} .",
    ]


def _sw_entity_hq_country_filter_lines(var_name: str, country: Dict[str, str]) -> List[str]:
    cq = country["qid"]
    v = var_name.lstrip("?")
    return [f"?{v} wdt:P159/wdt:P17 wd:{cq} ."]


def _sw_direct_developer_phrase(dev: Dict[str, str]) -> Tuple[str, str]:
    return (f"разработанных {dev['ru']}", f"were developed by {dev['en']}")


def _sw_developer_hq_country_phrase(country: Dict[str, str]) -> Tuple[str, str]:
    return (
        f"разработанных организацией со штаб-квартирой в {country['ru']}",
        f"were developed by an organization headquartered in {country['en']}",
    )


def _sw_presence_phrase(presence: str) -> Tuple[str, str]:
    if presence == "website":
        return "у которых указан официальный сайт", "have an official website recorded"
    if presence == "repo":
        return "у которых указан репозиторий исходного кода", "have a source code repository recorded"
    return "", ""


# Concrete L5 candidates. They are deliberately explicit and numerous, because WDQS is sparse.
# stage="hard" candidates use strong hidden license/developer bridges. stage="balanced" candidates
# reintroduce same-language/same-OS only with enough additional direct filters to be L5-worthy.
SOFTWARE_L5_CANDIDATES: List[Dict[str, Any]] = []


def _sw_add_l5_candidate(**kw) -> None:
    SOFTWARE_L5_CANDIDATES.append(dict(kw))


# Strong DBMS and framework candidates around Apache-style licenses.
for seed in ["Android", "TensorFlow"]:
    for year_from, year_to in [(1995, 2015), (2000, 2024), (2005, 2024)]:
        _sw_add_l5_candidate(stage="hard", bridge="same_license", kind="database_management_system", seed=seed,
                             developer="Apache Software Foundation", language="Java", year_from=year_from, year_to=year_to,
                             presence=["repo", "website"])
        # Frameworks are allowed only in the strong Apache-specific pattern; generic framework L1-L3 was noisy,
        # but this L5 query is constrained enough and gives useful hard examples.
        _sw_add_l5_candidate(stage="hard", bridge="same_license", kind="software_framework", seed=seed,
                             developer="Apache Software Foundation", language="Java", year_from=year_from, year_to=year_to,
                             presence=["repo", "website"])

# Strong browser candidates: license bridge + direct OS + direct language + year + presence.
for seed in ["Mozilla Firefox", "Git", "VLC media player", "Android", "Chromium"]:
    for os_name in ["Microsoft Windows", "Linux", "macOS"]:
        for year_from, year_to in [(1995, 2024), (2000, 2024), (2010, 2024)]:
            _sw_add_l5_candidate(stage="hard", bridge="same_license", kind="web_browser", seed=seed,
                                 os=os_name, language="C++", year_from=year_from, year_to=year_to,
                                 presence=["repo", "website"])

# Strong text-editor/source-code-editor candidates.
for seed in ["Git", "Mozilla Firefox", "Android", "TensorFlow", "WordPress", "VLC media player"]:
    for os_name in ["Microsoft Windows", "Linux", "macOS"]:
        for lang in ["C++", "Python", "JavaScript", "TypeScript"]:
            for year_from, year_to in [(2000, 2024), (2010, 2024)]:
                _sw_add_l5_candidate(stage="hard", bridge="same_license", kind="text_editor", seed=seed,
                                     os=os_name, language=lang, year_from=year_from, year_to=year_to,
                                     presence=["repo", "website"])

# Strong IDE candidates.
for seed in ["Android", "TensorFlow", "Mozilla Firefox", "Git"]:
    for os_name in ["Microsoft Windows", "Linux", "macOS"]:
        for lang in ["Python", "C++", "JavaScript"]:
            _sw_add_l5_candidate(stage="hard", bridge="same_license", kind="integrated_development_environment", seed=seed,
                                 os=os_name, language=lang, year_from=2000, year_to=2024,
                                 presence=["repo", "website"])

# Same-developer bridge, restricted to Apple OS family. HQ is only used here because bridgeDeveloper=Apple is explicit in WDQS.
for lang, y0, y1 in [("Swift", 2007, 2024), ("C", 2000, 2024), ("C++", 2000, 2024)]:
    _sw_add_l5_candidate(stage="hard", bridge="same_developer", kind="operating_system", seed="macOS",
                         language=lang, hq_country="United States", year_from=y0, year_to=y1,
                         presence=["website"])

# Balanced fallback: weak bridge is allowed only with direct filters + year + website/repo.
# These are intentionally limited and get their own families, so they cannot dominate.
for seed in ["TensorFlow", "Mozilla Firefox", "WordPress", "Android", "macOS", "VLC media player"]:
    for kind, os_names, license_names, lang_names in [
        ("text_editor", ["Microsoft Windows", "Linux", "macOS"], ["MIT License", "GNU General Public License"], []),
        ("web_browser", ["Microsoft Windows", "Linux", "macOS"], ["GNU General Public License", "Mozilla Public License"], []),
        ("integrated_development_environment", ["Microsoft Windows", "Linux", "macOS"], ["MIT License", "GNU General Public License"], []),
    ]:
        for os_name in os_names:
            for lic in license_names:
                _sw_add_l5_candidate(stage="balanced", bridge="same_language", kind=kind, seed=seed,
                                     os=os_name, license=lic, year_from=2010, year_to=2024,
                                     presence=["repo", "website"])

for seed in ["TensorFlow", "WordPress", "Android", "Mozilla Firefox"]:
    for kind, lang_names, license_names in [
        ("web_browser", ["C++", "JavaScript"], ["GNU General Public License", "Mozilla Public License", "MIT License"]),
        ("text_editor", ["C++", "Python", "JavaScript"], ["GNU General Public License", "MIT License"]),
        ("integrated_development_environment", ["Python", "C++", "JavaScript"], ["MIT License", "GNU General Public License"]),
    ]:
        for lang in lang_names:
            for lic in license_names:
                _sw_add_l5_candidate(stage="balanced", bridge="same_os", kind=kind, seed=seed,
                                     language=lang, license=lic, year_from=2010, year_to=2024,
                                     presence=["repo", "website"])


# Runtime blocklist to avoid repeating dead WDQS candidates thousands of times.
_SW_L5_BLOCKED_TEMPLATE_IDS: set[str] = set()
_SW_L5_FAILED_TEMPLATE_IDS: set[str] = set()


def _sw_l5_mark_blocked(record_or_template_id: Any, reason: str = "blocked") -> None:
    if isinstance(record_or_template_id, dict):
        tid = record_or_template_id.get("template_id")
    else:
        tid = str(record_or_template_id or "")
    if tid:
        _SW_L5_BLOCKED_TEMPLATE_IDS.add(tid)
        if reason and reason != "accepted":
            _SW_L5_FAILED_TEMPLATE_IDS.add(tid)


def _sw_l5_candidate_template_id(combo_i: int, bridge_type: str, presence: str, stage: str) -> str:
    return f"software_l5_v8_{stage}_{bridge_type}_{combo_i:03d}_{presence}"


def _sw_add_l5_year_constraint(where: List[str], ru_parts: List[str], en_parts: List[str], constraints: Dict[str, Any], kind: str, year_from: int, year_to: int) -> None:
    if kind == "programming_language":
        where += _sw_inception_year_lines(year_from, year_to)
        ru_y, en_y = _sw_inception_year_phrase(year_from, year_to)
    else:
        where += _sw_year_lines(year_from, year_to)
        ru_y, en_y = _sw_year_phrase(year_from, year_to)
    constraints.update({"year_from": int(year_from), "year_to": int(year_to)})
    ru_parts.append(ru_y)
    en_parts.append(en_y)


def _sw_build_one_l5_candidate(idx: int, combo_i: int, combo: Dict[str, Any], presence: str) -> Optional[BenchmarkExample]:
    bridge_type = combo["bridge"]
    kind = combo["kind"]
    stage = combo.get("stage", "hard")
    template_id = _sw_l5_candidate_template_id(combo_i, bridge_type, presence, stage)
    if template_id in _SW_L5_BLOCKED_TEMPLATE_IDS:
        globals()["_SW_L5_LAST_REJECT"] = "template_already_blocked"
        return None

    requested = _sw_requested_count("L5")
    where = _sw_kind_type_lines(kind)
    constraints: Dict[str, Any] = {"kind": kind}
    ru_parts: List[str] = []
    en_parts: List[str] = []
    bridge_meta: Optional[Dict[str, Any]] = None

    seed = _sw_by_en(SEED_SOFTWARE, combo["seed"])
    if bridge_type == "same_license":
        where += _sw_same_license_as_seed_lines(seed)
        constraints["license_from_software"] = seed["en"]
        ru_parts.append(f"имеющих хотя бы одну из тех же лицензий, что и «{seed['ru']}»")
        en_parts.append(f"share at least one license with \"{seed['en']}\"")
        bridge_meta = _sw_bridge_meta(bridge="license", constraint_key="license_from_software", property_id="P275", seed=seed)
    elif bridge_type == "same_developer":
        where += _sw_same_developer_as_seed_lines(seed)
        constraints["developer_from_software"] = seed["en"]
        ru_parts.append(f"разработанных тем же разработчиком, что и «{seed['ru']}»")
        en_parts.append(f"were developed by the same developer as \"{seed['en']}\"")
        bridge_meta = _sw_bridge_meta(bridge="developer", constraint_key="developer_from_software", property_id="P178", seed=seed)
    elif bridge_type == "same_language":
        where += _sw_same_programming_language_as_seed_lines(seed)
        constraints["programming_language_from_software"] = seed["en"]
        ru_parts.append(f"написанных хотя бы на одном из тех же языков программирования, что и «{seed['ru']}»")
        en_parts.append(f"are written in at least one of the same programming languages as \"{seed['en']}\"")
        bridge_meta = _sw_bridge_meta(bridge="programming_language", constraint_key="programming_language_from_software", property_id="P277", seed=seed)
    elif bridge_type == "same_os":
        where += _sw_same_operating_system_as_seed_lines(seed)
        constraints["operating_system_from_software"] = seed["en"]
        ru_parts.append(f"работающих хотя бы на одной из тех же операционных систем, что и «{seed['ru']}»")
        en_parts.append(f"run on at least one of the same operating systems as \"{seed['en']}\"")
        bridge_meta = _sw_bridge_meta(bridge="operating_system", constraint_key="operating_system_from_software", property_id="P306", seed=seed)
    else:
        _sw_l5_mark_blocked(template_id, "unknown_bridge")
        return None

    if "developer" in combo:
        dev = _sw_by_en(DEVELOPERS, combo["developer"])
        where.append(_sw_developer_line(dev))
        constraints["developer"] = dev["en"]
        ru_d, en_d = _sw_direct_developer_phrase(dev)
        ru_parts.append(ru_d)
        en_parts.append(en_d)

    if "os" in combo:
        os_item = _sw_by_en(OPERATING_SYSTEMS, combo["os"])
        where.append(_sw_operating_system_line(os_item))
        constraints["operating_system"] = os_item["en"]
        ru_parts.append(f"работающих на {os_item['ru']}")
        en_parts.append(f"run on {os_item['en']}")

    if "language" in combo:
        lang = _sw_by_en(PROGRAMMING_LANGUAGES, combo["language"])
        where.append(_sw_programming_language_line(lang))
        constraints["programming_language"] = lang["en"]
        ru_parts.append(f"написанных на {lang['ru']}")
        en_parts.append(f"are written in {lang['en']}")

    if "license" in combo:
        lic = _sw_by_en(LICENSES, combo["license"])
        where.append(_sw_license_line(lic))
        constraints["license"] = lic["en"]
        ru_l, en_l = _sw_license_phrase(lic)
        ru_parts.append(ru_l)
        en_parts.append(en_l)

    if "hq_country" in combo:
        country = _sw_by_en(COUNTRIES, combo["hq_country"])
        if bridge_type == "same_developer":
            where += _sw_entity_hq_country_filter_lines("bridgeDeveloper", country)
        else:
            where += _sw_developer_hq_country_lines(country, developer_var="developer_l5")
        constraints["developer_headquarters_country"] = country["en"]
        ru_c, en_c = _sw_developer_hq_country_phrase(country)
        ru_parts.append(ru_c)
        en_parts.append(en_c)

    _sw_add_l5_year_constraint(where, ru_parts, en_parts, constraints, kind, int(combo["year_from"]), int(combo["year_to"]))

    if presence == "website":
        where.append(_sw_has_official_website_line())
        constraints["has_official_website"] = True
    elif presence == "repo":
        where.append(_sw_has_source_repo_line())
        constraints["has_source_code_repository"] = True
    else:
        _sw_l5_mark_blocked(template_id, "bad_presence")
        return None
    ru_p, en_p = _sw_presence_phrase(presence)
    ru_parts.append(ru_p)
    en_parts.append(en_p)

    q_ru, q_en = _sw_finish_text(kind, requested, ru_parts, en_parts)
    ex = _sw_finalize_wdqs_example(
        idx=idx,
        complexity="L5",
        template_id=template_id,
        template_family=f"l5_v8_{stage}_{bridge_type}",
        query_text_ru=q_ru,
        query_text_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        bridge_meta=bridge_meta,
    )
    if ex is None:
        _sw_l5_mark_blocked(template_id, "no_example_or_insufficient_gold")
        globals()["_SW_L5_LAST_REJECT"] = "no_example_or_insufficient_gold"
        return None

    rec = _sw_asdict(ex)
    reject = _sw_l5_complexity_reject_reason(rec)
    if reject:
        _sw_l5_mark_blocked(template_id, reject)
        globals()["_SW_L5_LAST_REJECT"] = reject
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[SKIP] {template_id}: {reject}")
        return None

    globals()["_SW_L5_LAST_REJECT"] = None
    return ex


def _sw_tpl_l5_productive(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None

    # Try a small batch of concrete candidates in one generation attempt.
    # Prefer hard candidates first, then balanced fallback.
    order = list(range(len(SOFTWARE_L5_CANDIDATES)))
    rng.shuffle(order)
    order.sort(key=lambda i: 0 if SOFTWARE_L5_CANDIDATES[i].get("stage") == "hard" else 1)

    tried = 0
    last_reject = None
    for combo_i in order:
        combo = SOFTWARE_L5_CANDIDATES[combo_i]
        presences = list(combo.get("presence") or ["website"])
        rng.shuffle(presences)
        for presence in presences:
            template_id = _sw_l5_candidate_template_id(combo_i, combo["bridge"], presence, combo.get("stage", "hard"))
            if template_id in _SW_L5_BLOCKED_TEMPLATE_IDS:
                continue
            tried += 1
            ex = _sw_build_one_l5_candidate(idx, combo_i, combo, presence)
            if ex is not None:
                return ex
            last_reject = globals().get("_SW_L5_LAST_REJECT") or "candidate_failed"
            if tried >= globals().get("SOFTWARE_L5_MAX_CANDIDATE_TRIES_PER_ATTEMPT", 18):
                globals()["_SW_L5_LAST_REJECT"] = last_reject or "candidate_batch_failed"
                return None

    globals()["_SW_L5_LAST_REJECT"] = "all_l5_candidate_templates_exhausted"
    return None


SOFTWARE_L5_MAX_CANDIDATE_TRIES_PER_ATTEMPT = 18
print(f"L5 v8 candidate templates: {sum(len(c.get('presence') or ['website']) for c in SOFTWARE_L5_CANDIDATES)}")


L5 v8 candidate templates: 837


In [8]:
SOFTWARE_TEMPLATE_REGISTRY: Dict[str, List[Tuple[Callable[[str, int, random.Random], Optional[BenchmarkExample]], float]]] = {
    "L1": [
        (_sw_tpl_l1_narrow_by_developer, 1.00),
        (_sw_tpl_l1_narrow_by_language, 1.25),
        (_sw_tpl_l1_narrow_by_license, 0.95),
        (_sw_tpl_l1_web_browser_by_os, 0.95),
        (_sw_tpl_l1_operating_system_by_language, 0.80),
    ],
    "L2": [
        (_sw_tpl_l2_narrow_language_license, 1.25),
        (_sw_tpl_l2_narrow_language_year, 1.10),
        (_sw_tpl_l2_narrow_devcountry_license, 1.05),
        (_sw_tpl_l2_runnable_os_license, 1.00),
        (_sw_tpl_l2_web_browser_os_language, 0.95),
        (_sw_tpl_l2_os_language_year, 0.85),
    ],
    "L3": [
        (_sw_tpl_same_developer_language, 1.10),
        (_sw_tpl_same_language_os_license, 1.05),
        (_sw_tpl_developer_country_language_year, 1.10),
        (_sw_tpl_license_language_os_devcountry, 1.00),
        (_sw_tpl_web_browser_devcountry_language_license, 0.90),
        (_sw_tpl_same_license_as_seed_language_country, 0.95),
        (_sw_tpl_same_os_as_seed_license_language, 0.95),
    ],
    "L4": [
        (_sw_tpl_same_developer_language, 1.00),
        (_sw_tpl_same_language_os_license, 1.00),
        (_sw_tpl_developer_country_language_year, 1.10),
        (_sw_tpl_license_language_os_devcountry, 1.05),
        (_sw_tpl_operating_system_devcountry_language, 0.95),
        (_sw_tpl_operating_system_same_developer_as_seed, 0.80),
        (_sw_tpl_web_browser_devcountry_language_license, 0.95),
        (_sw_tpl_programming_language_same_creator_as_seed, 0.70),
        (_sw_tpl_same_license_as_seed_language_country, 1.00),
        (_sw_tpl_same_os_as_seed_license_language, 1.00),
    ],
    "L5": [
        (_sw_tpl_l5_productive, 1.00),
    ],
}


def _sw_choose_template(complexity: str, rng: random.Random):
    entries = SOFTWARE_TEMPLATE_REGISTRY.get(complexity, [])
    funcs = [x[0] for x in entries]
    weights = [x[1] for x in entries]
    return rng.choices(funcs, weights=weights, k=1)[0]


def _sw_template_family_allowed(records: List[Dict[str, Any]], record: Dict[str, Any]) -> bool:
    family = record.get("template_family")
    template_id = record.get("template_id")
    level = record.get("complexity")
    if not family or not template_id:
        return True

    total_family = sum(1 for r in records if r.get("template_family") == family)
    level_family = sum(1 for r in records if r.get("template_family") == family and r.get("complexity") == level)
    total_tpl = sum(1 for r in records if r.get("template_id") == template_id)
    level_tpl = sum(1 for r in records if r.get("template_id") == template_id and r.get("complexity") == level)

    if total_family >= SOFTWARE_MAX_PER_TEMPLATE_FAMILY_TOTAL:
        return False
    if level_family >= SOFTWARE_MAX_PER_TEMPLATE_FAMILY_LEVEL:
        return False
    if total_tpl >= SOFTWARE_MAX_PER_EXACT_TEMPLATE_TOTAL:
        return False
    if level_tpl >= SOFTWARE_MAX_PER_EXACT_TEMPLATE_LEVEL:
        return False
    return True


def _sw_basic_record_quality_ok(record: Dict[str, Any]) -> Tuple[bool, str]:
    required_keys = list(asdict(BenchmarkExample(
        id="x", domain="x", complexity="L1", query_text_ru="x", constraints={}, requested_count=1,
        gold_answer_qids=[], gold_answer_labels_ru=[], sparql_query="x", created_at="x",
    )).keys())
    missing = [k for k in required_keys if k not in record]
    if missing:
        return False, f"missing_keys:{missing}"
    if not record.get("query_text_ru") or not record.get("query_text_en"):
        return False, "missing_query_text"
    if not isinstance(record.get("constraints"), dict) or not record["constraints"]:
        return False, "bad_constraints"
    bad_constraint_keys = [k for k in record["constraints"] if k.endswith("_qid") or k.endswith("_ru") or k in BAD_CONSTRAINT_KEYS]
    if bad_constraint_keys:
        return False, f"bad_constraint_keys:{bad_constraint_keys}"
    if record["constraints"].get("kind") in SOFTWARE_DISABLED_KIND_KEYS:
        return False, f"disabled_or_broad_kind:{record['constraints'].get('kind')}"
    gold_count = len(record.get("gold_answer_qids") or [])
    requested = int(record.get("requested_count") or 0)
    if record.get("complexity") == "L5":
        if gold_count < requested:
            return False, "gold_count_below_requested_count"
    elif gold_count <= requested:
        return False, "gold_count_not_above_requested_count"
    if len(record.get("gold_answer_qids") or []) != len(record.get("gold_answer_labels_ru") or []):
        return False, "qid_ru_label_length_mismatch"
    if len(record.get("gold_answer_qids") or []) != len(record.get("gold_answer_labels_en") or []):
        return False, "qid_en_label_length_mismatch"
    if _sw_duplicate_labels(record.get("gold_answer_labels_ru") or []):
        return False, "duplicate_ru_gold_labels"
    if _sw_duplicate_labels(record.get("gold_answer_labels_en") or []):
        return False, "duplicate_en_gold_labels"
    for lab in (record.get("gold_answer_labels_en") or []) + (record.get("gold_answer_labels_ru") or []):
        if _sw_is_bad_gold_label(str(lab or "")):
            return False, "bad_or_version_like_gold_label"
    l5_reason = _sw_l5_complexity_reject_reason(record)
    if l5_reason:
        return False, l5_reason
    if SOFTWARE_STRICT_FULL_GOLD and record.get("gold_truncated"):
        return False, "truncated_gold_in_strict_mode"
    return True, "ok"



def _sw_gold_signature(record: Dict[str, Any]) -> Tuple[str, ...]:
    return tuple(sorted(str(q) for q in (record.get("gold_answer_qids") or []) if q))


def _sw_load_reference_records(paths: List[Path]) -> List[Dict[str, Any]]:
    refs: List[Dict[str, Any]] = []
    seen_files = set()
    for p in paths:
        try:
            path = Path(p)
            key = str(path.resolve()) if path.exists() else str(path)
            if key in seen_files or not path.exists() or path == SOFTWARE_OUT_PATH:
                continue
            seen_files.add(key)
            refs.extend(_sw_read_jsonl(path))
        except Exception:
            continue
    return refs

def _sw_current_audit(records: List[Dict[str, Any]], skipped: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "domain": DOMAIN,
        "target_plan": TARGET_PLAN_SOFTWARE,
        "strict_full_gold": SOFTWARE_STRICT_FULL_GOLD,
        "records_total": len(records),
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in records)),
        "counts_by_template_family": dict(Counter(r.get("template_family") for r in records)),
        "counts_by_template_id": dict(Counter(r.get("template_id") for r in records)),
        "counts_by_kind": dict(Counter((r.get("constraints") or {}).get("kind") for r in records)),
        "skipped_count": len(skipped),
        "skipped_preview": skipped[-100:],
        "output_path": str(SOFTWARE_OUT_PATH),
        "audit_path": str(SOFTWARE_AUDIT_PATH),
        "checkpoint_path": str(SOFTWARE_CHECKPOINT_PATH),
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }



def _sw_existing_output_compatible(records: List[Dict[str, Any]]) -> Tuple[bool, str]:
    """Return False for stale outputs whose counts cannot lead to the current target plan."""
    if not records:
        return True, "empty"
    counts = Counter(r.get("complexity") for r in records)
    target_levels = set(TARGET_PLAN_SOFTWARE.keys())
    for level, count in counts.items():
        if level not in target_levels:
            return False, f"unexpected_level:{level}"
        if count > TARGET_PLAN_SOFTWARE.get(level, 0):
            return False, f"level_above_target:{level}:{count}>{TARGET_PLAN_SOFTWARE.get(level, 0)}"
    if len(records) > sum(TARGET_PLAN_SOFTWARE.values()):
        return False, "total_above_target"
    bad_domains = [r.get("domain") for r in records if r.get("domain") != DOMAIN]
    if bad_domains:
        return False, f"unexpected_domain:{bad_domains[:3]}"
    return True, "compatible_partial_output"


def _sw_backup_existing_output_files(reason: str) -> None:
    ts = time.strftime("%Y%m%d_%H%M%S")
    for path in [SOFTWARE_OUT_PATH, SOFTWARE_AUDIT_PATH, SOFTWARE_CHECKPOINT_PATH]:
        if not path.exists():
            continue
        backup = path.with_name(f"{path.name}.bak_{ts}_{reason}")
        path.replace(backup)
        print(f"[INFO] backed up {path} -> {backup}")



def _sw_rewrite_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()


def _sw_prune_existing_records_for_quality(records: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    kept, dropped = [], []
    for r in records:
        ok, reason = _sw_basic_record_quality_ok(r)
        if ok:
            kept.append(r)
        else:
            dropped.append({"id": r.get("id"), "complexity": r.get("complexity"), "template_id": r.get("template_id"), "reason": reason})
    return kept, dropped



def _sw_gold_set(record: Dict[str, Any]) -> set[str]:
    return {str(q) for q in (record.get("gold_answer_qids") or []) if q}


def _sw_gold_overlap_reject_reason(record: Dict[str, Any], previous_records: List[Dict[str, Any]]) -> Optional[str]:
    """Reject near-duplicate tasks whose answer set heavily overlaps existing/reference data."""
    new = _sw_gold_set(record)
    if not new:
        return None
    for old in previous_records:
        old_set = _sw_gold_set(old)
        if not old_set:
            continue
        inter = len(new & old_set)
        if inter < globals().get("SOFTWARE_MIN_SHARED_GOLD_FOR_OVERLAP_CHECK", 3):
            continue
        union = len(new | old_set)
        jaccard = inter / union if union else 0.0
        containment = inter / max(1, min(len(new), len(old_set)))
        if jaccard >= globals().get("SOFTWARE_MAX_GOLD_JACCARD_OVERLAP", 0.70):
            return f"near_duplicate_gold_jaccard:{jaccard:.2f}:with:{old.get('id')}"
        if containment >= globals().get("SOFTWARE_MAX_GOLD_CONTAINMENT_OVERLAP", 0.84):
            return f"near_duplicate_gold_containment:{containment:.2f}:with:{old.get('id')}"
    return None

def generate_software_dataset() -> List[Dict[str, Any]]:
    rng = random.Random(SOFTWARE_SEED)
    records = _sw_read_jsonl(SOFTWARE_OUT_PATH)

    skipped: List[Dict[str, Any]] = []

    if records and SOFTWARE_FORCE_FRESH_RUN:
        _sw_backup_existing_output_files("forced_fresh_run")
        records = []
    elif records and SOFTWARE_AUTOBACKUP_INCOMPATIBLE_OUTPUT:
        compatible, reason = _sw_existing_output_compatible(records)
        if not compatible:
            print(f"[INFO] existing software output is incompatible with current target plan: {reason}")
            _sw_backup_existing_output_files("incompatible_target_plan")
            records = []

    if records and SOFTWARE_PRUNE_EXISTING_OUTPUT:
        kept, dropped = _sw_prune_existing_records_for_quality(records)
        if dropped:
            print(f"[INFO] pruning {len(dropped)} existing records that fail current quality gates")
            _sw_backup_existing_output_files("quality_prune_before_resume")
            records = kept
            _sw_rewrite_jsonl(SOFTWARE_OUT_PATH, records)
            skipped.extend({**d, "stage": "existing_prune"} for d in dropped)

    seen_keys = {_sw_record_key(r) for r in records}
    reference_records = _sw_load_reference_records(globals().get("SOFTWARE_REFERENCE_JSONL_PATHS", []))
    seen_gold_signatures = {_sw_gold_signature(r) for r in records + reference_records if _sw_gold_signature(r)}
    seen_gold_records = list(records) + list(reference_records)
    if reference_records:
        print(f"reference records for exact-gold dedupe: {len(reference_records)}")

    existing_counts = Counter(r.get("complexity") for r in records)
    print("target plan:", TARGET_PLAN_SOFTWARE)
    print("existing records:", len(records))
    print("existing counts:", dict(existing_counts))

    total_target = sum(TARGET_PLAN_SOFTWARE.values())
    already_done_total = sum(min(existing_counts.get(level, 0), target) for level, target in TARGET_PLAN_SOFTWARE.items())

    overall_bar = tqdm(total=total_target, initial=already_done_total, desc="software total") if tqdm else None
    idx = len(records) + 1

    try:
        for complexity, target_n in TARGET_PLAN_SOFTWARE.items():
            if target_n <= 0:
                continue
            already_done = Counter(r.get("complexity") for r in records).get(complexity, 0)
            if already_done >= target_n:
                print(f"SKIP: {complexity} already has {already_done}/{target_n}")
                continue

            level_bar = tqdm(total=target_n, initial=already_done, desc=f"software:{complexity}", leave=True) if tqdm else None
            attempts = 0
            no_accept_streak = 0
            max_attempts = SOFTWARE_MAX_ATTEMPTS_BY_LEVEL.get(complexity, 4000)
            no_accept_limit = SOFTWARE_NO_ACCEPT_STREAK_LIMIT_BY_LEVEL.get(complexity, 600)

            while Counter(r.get("complexity") for r in records).get(complexity, 0) < target_n and attempts < max_attempts:
                attempts += 1
                func = _sw_choose_template(complexity, rng)
                if level_bar and attempts % SOFTWARE_PROGRESS_UPDATE_EVERY_ATTEMPTS == 0:
                    level_bar.set_postfix({"attempts": attempts, "no_accept": no_accept_streak, "tpl": getattr(func, "__name__", "")[:22]})
                if overall_bar and attempts % SOFTWARE_PROGRESS_UPDATE_EVERY_ATTEMPTS == 0:
                    overall_bar.set_postfix({"level": complexity, "attempts": attempts, "no_accept": no_accept_streak})
                if no_accept_streak >= no_accept_limit:
                    msg = f"no accepted {complexity} record for {no_accept_streak} attempts; moving on to avoid a long stall"
                    print(f"[WARN] {msg}")
                    skipped.append({"complexity": complexity, "reason": msg, "attempts": attempts})
                    break

                try:
                    globals()["_SW_L5_LAST_REJECT"] = None
                    ex = func(complexity, idx, rng)
                except Exception as e:
                    skipped.append({"complexity": complexity, "template": getattr(func, "__name__", str(func)), "reason": f"exception:{type(e).__name__}:{str(e)[:160]}"})
                    no_accept_streak += 1
                    if level_bar and attempts % 25 == 0:
                        level_bar.set_postfix({"attempts": attempts, "last": "exception"})
                    continue

                if ex is None:
                    no_accept_streak += 1
                    if attempts % 50 == 0:
                        skipped.append({"complexity": complexity, "template": getattr(func, "__name__", str(func)), "reason": globals().get("_SW_L5_LAST_REJECT") or "no_example_or_insufficient_gold"})
                    if level_bar and attempts % 25 == 0:
                        level_bar.set_postfix({"attempts": attempts, "last": "none"})
                    continue

                record = _sw_asdict(ex)
                key = _sw_record_key(record)
                if key in seen_keys:
                    no_accept_streak += 1
                    try:
                        _sw_l5_mark_blocked(record, "duplicate")
                    except Exception:
                        pass
                    skipped.append({"complexity": complexity, "template_id": record.get("template_id"), "reason": "duplicate"})
                    continue

                if not _sw_template_family_allowed(records, record):
                    no_accept_streak += 1
                    try:
                        _sw_l5_mark_blocked(record, "template_diversity_guard")
                    except Exception:
                        pass
                    skipped.append({"complexity": complexity, "template_id": record.get("template_id"), "template_family": record.get("template_family"), "reason": "template_diversity_guard"})
                    continue

                gold_sig = _sw_gold_signature(record)
                if gold_sig and gold_sig in seen_gold_signatures:
                    no_accept_streak += 1
                    try:
                        _sw_l5_mark_blocked(record, "exact_gold_duplicate_with_existing_or_reference")
                    except Exception:
                        pass
                    skipped.append({"complexity": complexity, "template_id": record.get("template_id"), "reason": "exact_gold_duplicate_with_existing_or_reference"})
                    continue

                if globals().get("SOFTWARE_ENABLE_NEAR_GOLD_OVERLAP_FILTER", True):
                    overlap_reason = _sw_gold_overlap_reject_reason(record, seen_gold_records)
                    if overlap_reason:
                        no_accept_streak += 1
                        try:
                            _sw_l5_mark_blocked(record, overlap_reason)
                        except Exception:
                            pass
                        skipped.append({"complexity": complexity, "template_id": record.get("template_id"), "reason": overlap_reason})
                        continue

                ok, reason = _sw_basic_record_quality_ok(record)
                if not ok:
                    no_accept_streak += 1
                    try:
                        _sw_l5_mark_blocked(record, reason)
                    except Exception:
                        pass
                    skipped.append({"complexity": complexity, "template_id": record.get("template_id"), "reason": reason})
                    continue

                try:
                    _sw_l5_mark_blocked(record, "accepted")
                except Exception:
                    pass
                _sw_append_jsonl(SOFTWARE_OUT_PATH, record)
                no_accept_streak = 0
                records.append(record)
                seen_keys.add(key)
                if gold_sig:
                    seen_gold_signatures.add(gold_sig)
                seen_gold_records.append(record)
                idx += 1

                if level_bar:
                    level_bar.update(1)
                    level_bar.set_postfix({"attempts": attempts, "tpl": record.get("template_family")})
                if overall_bar:
                    overall_bar.update(1)

                # Incremental audit/checkpoint on every accepted example.
                _sw_write_json(SOFTWARE_AUDIT_PATH, _sw_current_audit(records, skipped))
                _sw_write_json(SOFTWARE_CHECKPOINT_PATH, {
                    "last_complexity": complexity,
                    "accepted_records": len(records),
                    "attempts_current_level": attempts,
                    "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                })

            if level_bar:
                level_bar.close()

            got = Counter(r.get("complexity") for r in records).get(complexity, 0)
            if got < target_n:
                print(f"[WARN] only generated {got}/{target_n} for {complexity} after {attempts} attempts")

    finally:
        if overall_bar:
            overall_bar.close()
        _sw_write_json(SOFTWARE_AUDIT_PATH, _sw_current_audit(records, skipped))

    print("final counts:", dict(Counter(r.get("complexity") for r in records)))
    print("template families:", dict(Counter(r.get("template_family") for r in records)))
    print("output:", SOFTWARE_OUT_PATH.resolve())
    return records


if RUN_SOFTWARE_GENERATION:
    software_records = generate_software_dataset()
else:
    print("RUN_SOFTWARE_GENERATION=False; definitions loaded but generation skipped.")


target plan: {'L1': 0, 'L2': 0, 'L3': 0, 'L4': 5, 'L5': 30}
existing records: 0
existing counts: {}


software:L4:  80%|████████  | 4/5 [20:05<05:01, 301.29s/it, attempts=1560, no_accept=698, tpl=_sw_tpl_same_os_as_see]


[WARN] no accepted L4 record for 700 attempts; moving on to avoid a long stall
[WARN] only generated 4/5 for L4 after 1562 attempts


software total:  40%|████      | 14/35 [20:22<30:34, 87.33s/it, level=L5, attempts=15030, no_accept=14992]

[WARN] no accepted L5 record for 15000 attempts; moving on to avoid a long stall
[WARN] only generated 10/30 for L5 after 15038 attempts
final counts: {'L4': 4, 'L5': 10}
template families: {'same_programming_language': 1, 'programming_language_same_creator': 1, 'same_operating_system': 1, 'operating_system_same_developer': 1, 'l5_v8_hard_same_developer': 2, 'l5_v8_hard_same_license': 8}
output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/software0.jsonl


## Post-generation schema/format validation

In [9]:
def validate_software_jsonl(path: Path = SOFTWARE_OUT_PATH) -> Dict[str, Any]:
    records = _sw_read_jsonl(path)
    expected_keys = list(asdict(BenchmarkExample(
        id="x", domain="x", complexity="L1", query_text_ru="x", constraints={}, requested_count=1,
        gold_answer_qids=[], gold_answer_labels_ru=[], sparql_query="x", created_at="x",
    )).keys())

    problems = []
    for i, r in enumerate(records, 1):
        keys = list(r.keys())
        if keys != expected_keys:
            problems.append({"line": i, "type": "schema_key_order_mismatch", "keys": keys})
            continue
        ok, reason = _sw_basic_record_quality_ok(r)
        if not ok:
            problems.append({"line": i, "type": "quality", "reason": reason})
        if r.get("domain") != DOMAIN:
            problems.append({"line": i, "type": "domain", "value": r.get("domain")})
        if r.get("complexity") not in {"L1", "L2", "L3", "L4", "L5"}:
            problems.append({"line": i, "type": "complexity", "value": r.get("complexity")})
        if any(k.endswith("_qid") or k.endswith("_ru") for k in (r.get("constraints") or {})):
            problems.append({"line": i, "type": "dirty_constraints", "constraints": r.get("constraints")})

    summary = {
        "path": str(path),
        "records": len(records),
        "counts_by_complexity": dict(Counter(r.get("complexity") for r in records)),
        "counts_by_template_family": dict(Counter(r.get("template_family") for r in records)),
        "counts_by_kind": dict(Counter((r.get("constraints") or {}).get("kind") for r in records)),
        "expected_key_order": expected_keys,
        "problem_count": len(problems),
        "problems_preview": problems[:50],
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

# Run after generation to verify exact JSONL format.
if SOFTWARE_OUT_PATH.exists():
    software_validation_summary = validate_software_jsonl(SOFTWARE_OUT_PATH)
else:
    print("No output file yet.")


{
  "path": "out_wikidata_benchmark/domain_outputs/software0.jsonl",
  "records": 14,
  "counts_by_complexity": {
    "L4": 4,
    "L5": 10
  },
  "counts_by_template_family": {
    "same_programming_language": 1,
    "programming_language_same_creator": 1,
    "same_operating_system": 1,
    "operating_system_same_developer": 1,
    "l5_v8_hard_same_developer": 2,
    "l5_v8_hard_same_license": 8
  },
  "counts_by_kind": {
    "text_editor": 2,
    "programming_language": 1,
    "operating_system": 3,
    "software_framework": 4,
    "web_browser": 3,
    "database_management_system": 1
  },
  "expected_key_order": [
    "id",
    "domain",
    "complexity",
    "query_text_ru",
    "constraints",
    "requested_count",
    "gold_answer_qids",
    "gold_answer_labels_ru",
    "sparql_query",
    "created_at",
    "query_text_en",
    "gold_answer_labels_en",
    "is_advanced",
    "template_id",
    "template_family",
    "gold_truncated",
    "ask_validator_sparql",
    "local_valida